# Piece of my Heart

## Overview

This module implements the piece system for the hex-based strategy game, including:
- **Piece simulation** - Calculate future movement paths based on instruction rules
- **Visual overlays** - Render piece positions and planned movements on the map
- **Interactive controls** - Facing selector widget for piece orientation
- **Route handlers** - Web endpoints for piece detail views and maps

## Architecture

### Three-Layer Design

1. **Kingdom Directive** — Strategic intent (FEED, EXPLORE, EXPAND, DEFEND, ATTACK)
2. **Piece Orders** — Directive mapped onto each piece based on type and stats
3. **Moves** — Resolved outcomes after all kingdoms' orders collide


I am going to work through some thoughts

## Modified Gosper
so the rules on Piece is going to be reworked as a list of ints with the following
0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
4 -> settle or siege # reserved for figuring out how to conquer settlements
5 -> forward # move on place forward (depending on movement cost). If you can't move forward (ie you hit water) rotate left. if it is there is another piece in the way pause   
6 -> rotate left
7 -> rotate right


## Movement costs

we have a movement cost formula
```python
@patch
def movement_cost(self: GameBoard, from_pos: HexPosition, to_pos: HexPosition, origin: int) -> float:
    """Calculate cost to move from one hex to another using HexPositions.
    
    Args:
        from_pos: Starting HexPosition
        to_pos: Destination HexPosition
        origin: Origin index for converting HexPosition to grid index
    
    Returns infinity for invalid moves (water, out of bounds).
    """
    terrain = self.terrain
    grid = terrain.hexGrid
    
    # Convert to indices to check terrain
    to_idx = grid.hexposition_to_index(to_pos, origin)
    from_idx = grid.hexposition_to_index(from_pos, origin)
    
    # Check bounds
    if to_idx < 0 or to_idx >= len(terrain.elevations):
        return float('inf')
    
    # Can't cross water
    if terrain.elevations[to_idx] < 1:
        return float('inf')
    
    from_elev = terrain.elevations[from_idx]
    to_elev = terrain.elevations[to_idx]
    
    # Base cost
    base_cost = 1.0
    
    # Elevation gain penalty (climbing is expensive)
    elev_diff = to_elev - from_elev
    if elev_diff > 0:
        # Exponential penalty for climbing
        base_cost += elev_diff * 2.0
    
    return base_cost
```
Which we will need to figure out how to use for a turn. movement_range will tell how many moves we can do. Lets make turns free (including turns from hitting the ocean). the first 4 counts as a single movement cost.

## food and diet
we are going to add three fields 
1. which is the amount of food a piece has
2. its diet. how much does it eat per turn If it rounds out of food it will start to lose health. 
3. food capacity which is how much total food it can carry.

## deprecate
we don't need memory since that will be in rules.

## settle
We are going to keep these around even though I haven't figure them out
harvest_goal: Resources = None
settle_progress: int = 0
settle_threshold: int = 3
sight = 3

## The things we carry
Lets start thinking about the rules editor for a piece
1. we are going to want a list of rules that are assigned and be able to add and delete rows
2. we are going to want an edit point where we can insert new items
3. we are going to want a HexDragMap that we can add items. The path we make needs to be converted into instructions on how we get there.
4. we need an apply button and the number of rounds. This will be our main map showing where the piece is going to move each turn for the number of turns. I envision this as a series of arrows and circles.
a. arrows are for movement. straight line if single cost squiggle line if multiple cost. the color of the arrow is how aggressive it is.
b. circles are for the pause actions (0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
) I think red should be for defend.

eventually we are going to want a drop down of a list of common rule sets (patrol around a circle. go out and harvest and comeback, attack by first flanking left). 
we will certainly want to have a hex destination that returns a set of rules on how to get there (hello djisktra)

I need to be clearer on some definitons
1. `round` is for all player and all pieces. Each player gives the set of orders it wants its pieces to do. We need to build out how to carry them out, but I am tempted just to start with the kindergaren model of everybody does what it wants
2. `turn` is equivalent to a round, but just on a single piece. `movement_cost` is how many things it can do in a turn. Some (like rotation) are free. Others like mmoving up a hill might cost a couple of movements.

It can be the case that rule ends and there are spare items. So if the last item in the list is one of the non movement items. ie (like pause) then we just pause the remaining amount of time. if it is a movement (or rotate) then we circle back to the begining.

## Pulling Ranks
so things are prioritzed in chess order ie kings move first and eat first. then queens. This way if a king will move to a spot before a pawn would

I am going to have the flow of the hex be the amount you can harvest. Settlments are going to have build queues. I need to figure out the mechanics of adding settlements whether there are pre existing locations or it behavior on a item

**7. Food economy**

Ill need a `hex_food_yield(terrain, idx) -> float` keyed off climate and flow.



**3. Forward-blocked spin trap**

Instruction 4 says "rotate left if water, pause if piece." But imagine a piece in a cove with water on 5 sides — it rotates 5 times but each time if the chance to move becomes available



Here's where we've landed:

**Budget model** — `move_strength` (rename to `speed`) is the per-turn movement budget, not per-instruction.

| Instruction | Cost | Notes |
|---|---|---|
| `FORWARD` (5) | 1 flat/downhill, 2 uphill | Can't move if budget < cost → turn ends |
| `ROT_L` (6) / `ROT_R` (7) | free | Including auto-rotations from hitting water |
| `PAUSE` (0) | 1 | Explicit wait |
| `DEFEND` (1) | 1 | Boosts combat response |
| `HARVEST` (2) | 1 | Gathers `hex_food_yield` into `food` (capped at `food_capacity`) |
| `GIVE` (3) | 1 | Transfer food to adjacent piece or settlement |
| `SETTLE` (4) | 1 | Reserved — siege/founding |

**FORWARD special cases:**
- **Water/edge** → auto-rotate left (free), up to 5×. Full circle → stuck, forced PAUSE (costs 1)
- **Occupied hex** → forced PAUSE (costs 1), cursor still advances

**End-of-rules:**
- Last instruction is non-movement (`PAUSE`/`DEFEND`/`HARVEST`/`GIVE`/`SETTLE`) → idle out remaining budget
- Last instruction is movement/rotation → wrap cursor to 0, keep going

**Turn resolution order:**
- KING → QUEEN → BISHOP → KNIGHT → ROOK → PAWN (chess rank priority)

**Food economy:**
- `food` — current supply
- `diet` — consumed per turn (deducted at turn end)
- `food_capacity` — max carried
- `food < 0` → lose health

**Example: Pawn (speed=3), rules `F F F F F F`, flat terrain:**
```
Turn 1: F(1) F(1) F(1) → budget 0
Turn 2: F(1) F(1) F(1) → budget 0
```

**Example: Pawn (speed=3), rules `F F F F F F`, uphill at step 2:**
```
Turn 1: F(1) F_uphill(2) → budget 0 (only 2 hexes moved)
Turn 2: F(1) F(1) F(1)  → budget 0
```

**Example: Pawn (speed=3), rules `F F HARVEST`:**
```
Turn 1: F(1) F(1) HARVEST(1) → end of rules, last=HARVEST → idle
Turn 2: same pattern (wraps because patrol=True)
```



In [ ]:
#| default_exp game/piece

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
import numpy as np
from dataclasses import dataclass
from datetime import datetime

In [ ]:
#| export
from monsterui.all import *

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion



In [ ]:
from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList

## Piece View — Project Plan
That's a really clean three-layer design:

1. **Kingdom Directive** — strategic intent ("we're starving", "explore", "expand")
2. **Piece Orders** — the directive gets mapped onto each piece based on its type and base stats
3. **Moves** — resolved outcomes after all kingdoms' orders collide

So the directive acts like the queen bee's pheromone signal — it shifts *what* each piece does, but *how well* they do it still depends on their base stats. A "gather food" directive would assign harvest orders to pawns (who are great at it) but might also reassign knights to patrol the borders instead of scouting.

For the directive enum, something like:

- **FEED** — prioritize harvest
- **EXPLORE** — prioritize scouting / vision
- **EXPAND** — claim new territory
- **DEFEND** — protect capital and settlements
- **ATTACK** — target enemy pieces/settlements

These look right. So we would to combine them to make a patrol concept. we could have a gosper curve that a piece would through as it moves. if we see something and it wants to attack then it could

That's a brilliant use of the Gosper curve — it's space-filling, so a piece on patrol would efficiently cover an area without a lot of backtracking. The concept would be something like:

    Generate patrol path — a Gosper curve anchored to a settlement or border, sized to fit the piece's territory
    Each turn — the piece advances along the curve by move_strength hexes
    At each hex — check sight range for triggers (enemy pieces, unclaimed resources, etc.)
    Interrupt — if something is spotted, the piece breaks off patrol and generates a new order (attack, harvest, scout closer)

So a patrol is really a streaming order — it keeps going until something more interesting happens. The directive level would control where the patrol is anchored and what triggers a break:

    DEFEND → patrol around capital, interrupt on enemy sighting → attack
    EXPLORE → patrol outward from borders, interrupt on discovery → report back
    FEED → patrol farmable land, interrupt on good hex → harvest


Nice. So let's sketch the Order shape. Something like:
Copied!

class OrderType(Enum):
    PATROL = "patrol"
    ATTACK = "attack"
    MOVE   = "move"
    HARVEST = "harvest"
    SCOUT  = "scout"




In [ ]:
app, rt, hexGameServer  = appRoutes()

## Helpers

In [ ]:
read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

In [ ]:
!cat ../../HexMagic/game/data.py

In [ ]:
#!cat ../../docs/ll*

### 1. Data Structures

In [ ]:
#| export
@dataclass
class PieceStep:
    """One resolved step in a piece's movement plan."""
    hex_idx: int
    facing: int
    instruction: Instruction
    prev_idx: int
    step: int               # global step counter (across all turns)
    turn: int               # which turn (0-indexed)
    cost: float             # movement points consumed by this step
    budget_remaining: float # budget left after this step
    blocked: bool = False
    auto_rotations: int = 0 # free rotations from hitting water/edge

    @staticmethod
    def steps_by_turn(steps: list['PieceStep']) -> dict[int, list['PieceStep']]:
        """Group steps by turn number."""
        turns = {}
        for s in steps:
            turns.setdefault(s.turn, []).append(s)
        return turns


# Cost categories
_COSTS_ONE = {Instruction.PAUSE, Instruction.DEFEND, Instruction.HARVEST,
              Instruction.GIVE, Instruction.SETTLE}
_FREE = {Instruction.ROT_L, Instruction.ROT_R}

def _move_cost(elevations, from_idx, to_idx) -> float:
    """Simple movement cost: 1 flat/downhill, 2 uphill."""
    diff = elevations[to_idx] - elevations[from_idx]
    return 2.0 if diff > 0 else 1.0


In [ ]:
#| export
@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_turns: int = 10,
             countries: np.ndarray = None,
             occupied: set[int] = None) -> list['PieceStep']:
    """Simulate piece over multiple turns using move_strength as per-turn budget.

    Budget rules:
      - FORWARD: costs _move_cost (1 flat, 2 uphill). If can't afford → end turn.
      - ROT_L / ROT_R: free
      - PAUSE / DEFEND / HARVEST / GIVE / SETTLE: cost 1
      - FORWARD into water/edge: auto-rotate left (free, up to 5×). Full circle → stuck, cost 1.
      - FORWARD into occupied hex: pause instead, cost 1.

    End-of-rules:
      - Last instruction was non-movement → pause out remaining budget.
      - Last instruction was movement/rotate → wrap cursor to 0.
    """
    il = self.instructions
    if not il.rules or self.location is None:
        return []

    occupied = set(occupied) if occupied else set()
    steps = []
    pos, fac, cur = self.location, self.facing, il.cursor
    step_n = 0

    for turn in range(num_turns):
        budget = float(self.move_strength)

        while budget > 0:
            # --- End of rules ---
            if cur >= len(il.rules):
                last = Instruction(il.rules[-1])
                if last in _COSTS_ONE:
                    # Idle out the turn
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE,
                                           pos, step_n, turn, budget, 0.0))
                    step_n += 1
                    break
                if il.patrol:
                    cur = 0  # wrap around
                else:
                    break    # one-shot: stop entirely

            instr = Instruction(il.rules[cur])
            prev = pos

            # --- FREE: rotations ---
            if instr in _FREE:
                fac = ((fac - 1) if instr == Instruction.ROT_L
                       else (fac + 1)) % 6
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 0.0, budget))
                step_n += 1; cur += 1
                continue

            # --- COST 1: non-movement actions ---
            if instr in _COSTS_ONE:
                budget -= 1.0
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 1.0, budget))
                step_n += 1; cur += 1
                continue

            # --- FORWARD ---
            auto_rots = 0
            moved = False
            saved_fac = fac

            while auto_rots <= 5:
                d = HexPosition.directions()[fac % 6]
                nbr = grid.hexposition_to_index(d, pos)

                passable = (0 <= nbr < len(elevations)
                            and nbr not in grid.invalidRegion
                            and elevations[nbr] >= 1)
                if passable and countries is not None:
                    passable = countries[nbr] >= 0

                if not passable:
                    fac = (fac - 1) % 6   # free auto-rotate left
                    auto_rots += 1
                    continue

                # Occupied → forced pause
                if nbr in occupied:
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                           step_n, turn, 1.0, budget - 1.0,
                                           blocked=True,
                                           auto_rotations=auto_rots))
                    budget -= 1.0
                    step_n += 1; cur += 1
                    moved = True
                    break

                # Can we afford it?
                cost = _move_cost(elevations, pos, nbr)
                if cost > budget:
                    budget = 0       # turn over, can't afford
                    moved = True
                    break

                # Move
                occupied.discard(pos)
                pos = nbr
                occupied.add(pos)
                budget -= cost
                steps.append(PieceStep(pos, fac, Instruction.FORWARD, prev,
                                       step_n, turn, cost, budget,
                                       auto_rotations=auto_rots))
                step_n += 1; cur += 1
                moved = True
                break

            if not moved:
                # Spun full circle — stuck
                fac = saved_fac
                budget = max(0, budget - 1.0)
                steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                       step_n, turn, 1.0, budget,
                                       blocked=True, auto_rotations=6))
                step_n += 1; cur += 1

    return steps


In [ ]:
#| export


@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_turns: int = 10, padding: int = 3,
                countries: np.ndarray = None,
                occupied: set[int] = None
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_turns,
                          countries=countries, occupied=occupied)
    visited = {self.location}
    for s in steps:
        visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations):
                expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)


### 2. Glyphs

In [ ]:
#| export
@dataclass
class DiagramGlyphs:
    """Shared SVG glyph renderer for piece-plan overlays."""
    color:        str   = "#333"
    stroke_width: float = 1.5
    opacity:      float = 0.85
    size:         float = 10.0


@patch
def circle(self: DiagramGlyphs, cx, cy, r, fill="none", stroke=None,
           stroke_width=None, opacity=None, **kw):
    stroke = stroke or self.color
    stroke_width = stroke_width or self.stroke_width
    opacity = opacity if opacity is not None else self.opacity
    return (f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{r:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="{stroke_width}" '
            f'opacity="{opacity}"/>\n')


@patch
def ring(self: DiagramGlyphs, cx, cy, r, stroke=None, stroke_width=2, opacity=0.9):
    return self.circle(cx, cy, r, fill="none",
                       stroke=stroke or self.color,
                       stroke_width=stroke_width, opacity=opacity)


@patch
def arc(self: DiagramGlyphs, cx, cy, r, start_deg, end_deg,
        stroke=None, stroke_width=2, opacity=0.8):
    import math
    stroke = stroke or self.color
    s, e = math.radians(start_deg), math.radians(end_deg)
    x1, y1 = cx + r * math.cos(s), cy + r * math.sin(s)
    x2, y2 = cx + r * math.cos(e), cy + r * math.sin(e)
    large = 1 if (end_deg - start_deg) % 360 > 180 else 0
    return (f'<path d="M {x1:.1f},{y1:.1f} A {r:.1f},{r:.1f} 0 {large} 1 '
            f'{x2:.1f},{y2:.1f}" fill="none" stroke="{stroke}" '
            f'stroke-width="{stroke_width}" opacity="{opacity}" '
            f'stroke-linecap="round"/>\n')


@patch
def shield(self: DiagramGlyphs, cx, cy, size=None, fill=None,
           stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    w, h = size, size * 1.2
    return (f'<path d="M {cx:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy+h*0.1:.1f} '
            f'Q {cx:.1f},{cy+h*0.5:.1f} {cx-w*0.5:.1f},{cy+h*0.1:.1f} '
            f'L {cx-w*0.5:.1f},{cy-h*0.5:.1f} Z" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
            f'opacity="{opacity}"/>\n')


@patch
def sheaf(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size
    svg = ""
    for dx in [-s*0.3, 0, s*0.3]:
        svg += (f'<line x1="{cx+dx:.1f}" y1="{cy-s*0.5:.1f}" '
                f'x2="{cx+dx*0.3:.1f}" y2="{cy+s*0.5:.1f}" '
                f'stroke="{fill}" stroke-width="{max(1.5, s*0.15):.1f}" '
                f'stroke-linecap="round" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx-s*0.35:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.35:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="{max(1, s*0.1):.1f}" '
            f'opacity="{opacity}"/>\n')
    return svg


@patch
def gift(self: DiagramGlyphs, cx, cy, size=None, fill=None,
         stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size * 0.8
    svg = (f'<rect x="{cx-s*0.5:.1f}" y="{cy-s*0.4:.1f}" '
           f'width="{s:.1f}" height="{s*0.8:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
           f'rx="{s*0.1:.1f}" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx:.1f}" y1="{cy-s*0.4:.1f}" '
            f'x2="{cx:.1f}" y2="{cy+s*0.4:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx-s*0.5:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.5:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    return svg


@patch
def house(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size
    svg = (f'<rect x="{cx-s*0.4:.1f}" y="{cy:.1f}" '
           f'width="{s*0.8:.1f}" height="{s*0.5:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
           f'opacity="{opacity}"/>\n')
    svg += (f'<polygon points="'
            f'{cx-s*0.5:.1f},{cy:.1f} '
            f'{cx:.1f},{cy-s*0.45:.1f} '
            f'{cx+s*0.5:.1f},{cy:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
            f'opacity="{opacity}"/>\n')
    return svg


@patch
def pause(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=0.7):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    w, h = size * 0.2, size * 0.6
    svg = ""
    for dx in [-size*0.18, size*0.18]:
        svg += (f'<rect x="{cx+dx-w/2:.1f}" y="{cy-h/2:.1f}" '
                f'width="{w:.1f}" height="{h:.1f}" '
                f'fill="{fill}" stroke="{stroke}" stroke-width="1" '
                f'rx="{w*0.2:.1f}" opacity="{opacity}"/>\n')
    return svg


@patch
def rot_arrow(self: DiagramGlyphs, cx, cy, r=None, clockwise=True,
              stroke=None, stroke_width=2, opacity=None):
    import math
    r = r or self.size * 0.5
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    if clockwise:
        arc_svg = self.arc(cx, cy, r, -120, 150, stroke, stroke_width, opacity)
        tip_x = cx + r * math.cos(math.radians(150))
        tip_y = cy + r * math.sin(math.radians(150))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x-r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x-r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    else:
        arc_svg = self.arc(cx, cy, r, 30, 300, stroke, stroke_width, opacity)
        tip_x = cx + r * math.cos(math.radians(30))
        tip_y = cy + r * math.sin(math.radians(30))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x+r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x+r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    return arc_svg + ah


In [ ]:
#| export
@patch
def blocked_x(self: DiagramGlyphs, cx, cy, r=None) -> str:
    """Red circle with white X cross — blocked/impassable move marker."""
    r = r or self.size * 0.25
    svg = self.circle(cx, cy, r, fill="red", stroke="darkred",
                      stroke_width=1, opacity=0.55)
    d = r * 0.55
    for x1, y1, x2, y2 in [(cx-d, cy-d, cx+d, cy+d),
                             (cx+d, cy-d, cx-d, cy+d)]:
        svg += (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
                f'x2="{x2:.1f}" y2="{y2:.1f}" '
                f'stroke="white" stroke-width="{max(1, r*0.35):.1f}" '
                f'stroke-linecap="round"/>\n')
    return svg


@patch
def start_marker(self: DiagramGlyphs, cx, cy, r=None,
                 stroke=None, stroke_width=None) -> str:
    """Hollow ring drawn around the piece's start position."""
    r = r or self.size * 0.8
    return self.ring(cx, cy, r,
                     stroke=stroke or self.color,
                     stroke_width=stroke_width or max(1.5, self.size * 0.07))


@patch
def step_label(self: DiagramGlyphs, cx, cy, n: int,
               offset=None, font_size=None, color=None) -> str:
    """Small step-number badge offset up-right from a hex centre.
    
    Args:
        n: step number (0-indexed internally; displayed as n+1)
        offset: pixel offset from centre (defaults to size * 0.35)
        font_size: override font size (defaults to max(12, size * 0.3))
        color: text color (defaults to self.comp if present, else self.color)
    """
    offset    = offset    or self.size * 0.35
    font_size = font_size or max(12, self.size * 0.3)
    color     = color     or getattr(self, 'comp', self.color)
    return (f'<text x="{cx+offset:.1f}" y="{cy-offset:.1f}" '
            f'text-anchor="start" font-size="{font_size:.0f}" '
            f'fill="{color}" font-family="sans-serif" '
            f'font-weight="bold">{n+1}</text>\n')


@patch
def action_glyph(self: DiagramGlyphs, instr: Instruction,
                 cx, cy, fill=None, stroke=None) -> str:
    """Dispatch a non-movement instruction to its geometry glyph.
    
    Returns empty string for movement instructions (FORWARD) — those
    are handled separately as arrows.
    """
    fill   = fill   or self.color
    stroke = stroke or getattr(self, 'comp', self.color)
    size   = self.size

    dispatch = {
        Instruction.PAUSE:   lambda: self.pause(cx, cy, size, fill, stroke),
        Instruction.DEFEND:  lambda: self.shield(cx, cy, size, fill, stroke),
        Instruction.HARVEST: lambda: self.sheaf(cx, cy, size, fill, stroke),
        Instruction.GIVE:    lambda: self.gift(cx, cy, size, fill, stroke),
        Instruction.SETTLE:  lambda: self.house(cx, cy, size, fill, stroke),
        Instruction.ROT_L:   lambda: self.rot_arrow(cx, cy, clockwise=False,
                                                     stroke=stroke),
        Instruction.ROT_R:   lambda: self.rot_arrow(cx, cy, clockwise=True,
                                                     stroke=stroke),
    }
    fn = dispatch.get(instr)
    return fn() if fn else ""


### 3. Instruction extensions

In [ ]:
#| export
INSTR_LABELS = {
    Instruction.PAUSE:   ("Pause",      "⏸"),
    Instruction.DEFEND:  ("Defend",     "🛡"),
    Instruction.HARVEST: ("Harvest",    "🍆"),
    Instruction.GIVE:    ("Give",       "🎁"),
    Instruction.SETTLE:  ("Settle",     "🏰"),
    Instruction.FORWARD: ("Forward",    "→"),
    Instruction.ROT_L:   ("Turn Left",  "↺"),
    Instruction.ROT_R:   ("Turn Right", "↻"),
}


def _instr_glyph(instr: Instruction, size=26, color="#444"):
    """Inline SVG glyph for one instruction."""
    g = DiagramGlyphs(color=color, size=size * 0.55)
    cx, cy = size / 2, size / 2
    if instr == Instruction.FORWARD:
        body = (f'<line x1="{cx-size*0.28:.1f}" y1="{cy:.1f}" '
                f'x2="{cx+size*0.2:.1f}" y2="{cy:.1f}" '
                f'stroke="{color}" stroke-width="2.5" stroke-linecap="round"/>'
                f'<polygon points="{cx+size*0.28:.1f},{cy:.1f} '
                f'{cx+size*0.12:.1f},{cy-size*0.13:.1f} '
                f'{cx+size*0.12:.1f},{cy+size*0.13:.1f}" fill="{color}"/>')
    else:
        body = g.action_glyph(instr, cx, cy) or ""
    return NotStr(f'<svg width="{size}" height="{size}" viewBox="0 0 {size} {size}" '
                  f'style="vertical-align:middle">{body}</svg>')


#| export
class InstructionList:
    """Displayable, serialisable list of piece instructions with glyphs and step numbers."""

    def __init__(self, rules: list[int], cursor: int = 0, patrol: bool = True):
        self.rules  = rules
        self.cursor = cursor
        self.patrol = patrol

    # ── Rendering ─────────────────────────────────────────────

    def __ft__(self):
        """Full MonsterUI table with SVG glyphs and cursor marker."""
        if not self.rules:
            return P("No instructions", cls=TextPresets.muted_sm)

        header = ["#", "", "Action"]
        rows   = []
        for i, r in enumerate(self.rules):
            instr = Instruction(r)
            label, _ = INSTR_LABELS.get(instr, (instr.name, "?"))
            step = Strong("▶ ", cls="text-primary") if i == self.cursor else str(i + 1)
            rows.append([step, _instr_glyph(instr), label])

        tbl  = TableFromLists(header, rows, cls=(TableT.sm, TableT.striped, TableT.hover))
        mode = "🔁 Patrol (loops)" if self.patrol else "⏹ One-shot"
        return Card(tbl, footer=Small(mode, cls=TextPresets.muted_sm))

    def compact(self, piece_id: str = "") -> FT:
        """Compact inline badge strip — lightweight alternative to the full table."""
        target_id = f"rules-{piece_id[:8]}" if piece_id else "rules-preview"

        if not self.rules:
            return P("No rules yet — drag a path above",
                     cls="text-sm opacity-50", id=target_id)

        _COLORS = {
            0: "#94a3b8", 1: "#ef4444", 2: "#22c55e",
            3: "#a855f7", 4: "#f59e0b", 5: "#3b82f6",
            6: "#64748b", 7: "#64748b",
        }
        badges = []
        for r in self.rules:
            instr  = Instruction(r)
            label, _ = INSTR_LABELS.get(instr, (instr.name, "?"))
            color  = _COLORS.get(r, "#888")
            badges.append(
                Span(label,
                     style=(f"background:{color}22; border:1px solid {color}; "
                            "border-radius:4px; padding:2px 6px; font-size:11px; "
                            f"color:{color}; margin:2px; display:inline-block;"))
            )

        return Div(
            P(f"{len(self.rules)} instructions", cls="text-xs opacity-50 mb-1"),
            Div(*badges, style="line-height:2"),
            id=target_id,
        )

    # ── Factories ─────────────────────────────────────────────

    @classmethod
    def from_path(cls, hex_indices: list[int], grid: HexGrid,
                  start_facing: int = 0, patrol: bool = True) -> "InstructionList":
        """Build an InstructionList directly from a dragged hex path."""
        return cls(
            rules=cls.path_to_rules(hex_indices, grid, start_facing),
            patrol=patrol,
        )

    @staticmethod
    def path_to_rules(hex_indices: list[int], grid: HexGrid,
                      start_facing: int = 0) -> list[int]:
        """Convert a list of hex grid indices into Instruction ints.

        Deduplicates consecutive repeated hexes (common in drag traces),
        then emits the minimum ROT_L / ROT_R turns needed before each FORWARD.
        Non-adjacent jumps are silently skipped.
        """
        if len(hex_indices) < 2:
            return []

        dirs = HexPosition.directions()   # [SW, W, NW, NE, E, SE] → indices 0–5

        # Remove consecutive duplicates from drag noise
        clean = [hex_indices[0]]
        for idx in hex_indices[1:]:
            if idx != clean[-1]:
                clean.append(idx)

        rules  = []
        facing = start_facing

        for i in range(len(clean) - 1):
            from_pos = grid.index_to_hexposition(clean[i],     clean[0])
            to_pos   = grid.index_to_hexposition(clean[i + 1], clean[0])
            delta    = to_pos - from_pos

            try:
                target_dir = next(j for j, d in enumerate(dirs) if d == delta)
            except StopIteration:
                continue   # non-adjacent hex — skip

            diff = (target_dir - facing) % 6
            if diff == 1:
                rules.append(Instruction.ROT_R.value)
            elif diff == 5:
                rules.append(Instruction.ROT_L.value)
            elif 2 <= diff <= 3:
                rules.extend([Instruction.ROT_R.value] * diff)
            elif diff == 4:
                rules.extend([Instruction.ROT_L.value] * 2)
            # diff == 0: already facing correctly

            rules.append(Instruction.FORWARD.value)
            facing = target_dir

        return rules


In [ ]:
# ── Rendering ─────────────────────────────────────────────
@patch
def __ft__(self: InstructionList):
    """Full MonsterUI table with SVG glyphs and cursor marker."""
    if not self.rules:
        return P("No instructions", cls=TextPresets.muted_sm)

    header = ["#", "", "Action"]
    rows   = []
    for i, r in enumerate(self.rules):
        instr = Instruction(r)
        label, _ = INSTR_LABELS.get(instr, (instr.name, "?"))
        step = Strong("▶ ", cls="text-primary") if i == self.cursor else str(i + 1)
        rows.append([step, _instr_glyph(instr), label])

    tbl  = TableFromLists(header, rows, cls=(TableT.sm, TableT.striped, TableT.hover))
    mode = "🔁 Patrol (loops)" if self.patrol else "⏹ One-shot"
    return Card(tbl, footer=Small(mode, cls=TextPresets.muted_sm))

@patch
def compact(self: InstructionList, piece_id: str = "") -> FT:
    """Compact inline badge strip — lightweight alternative to the full table."""
    target_id = f"rules-{piece_id[:8]}" if piece_id else "rules-preview"

    if not self.rules:
        return P("No rules yet — drag a path above",
                    cls="text-sm opacity-50", id=target_id)

    _COLORS = {
        0: "#94a3b8", 1: "#ef4444", 2: "#22c55e",
        3: "#a855f7", 4: "#f59e0b", 5: "#3b82f6",
        6: "#64748b", 7: "#64748b",
    }
    badges = []
    for r in self.rules:
        instr  = Instruction(r)
        label, _ = INSTR_LABELS.get(instr, (instr.name, "?"))
        color  = _COLORS.get(r, "#888")
        badges.append(
            Span(label,
                    style=(f"background:{color}22; border:1px solid {color}; "
                        "border-radius:4px; padding:2px 6px; font-size:11px; "
                        f"color:{color}; margin:2px; display:inline-block;"))
        )

    return Div(
        P(f"{len(self.rules)} instructions", cls="text-xs opacity-50 mb-1"),
        Div(*badges, style="line-height:2"),
        id=target_id,
    )

In [ ]:
#| export
@patch
def delete_at(self: InstructionList, idx: int):
    """Remove instruction at idx, clamping cursor."""
    if 0 <= idx < len(self.rules):
        self.rules.pop(idx)
        self.cursor = min(self.cursor, len(self.rules))

@patch
def insert_rules(self: InstructionList, new_rules: list[int]):
    """Insert rules at cursor position, advance cursor past them."""
    for i, r in enumerate(new_rules):
        self.rules.insert(self.cursor + i, r)
    self.cursor += len(new_rules)

@patch
def set_cursor(self: InstructionList, idx: int):
    self.cursor = max(0, min(idx, len(self.rules)))

@patch
def toggle_patrol(self: InstructionList):
    self.patrol = not self.patrol


In [ ]:
rules = [Instruction.FORWARD.value, Instruction.FORWARD.value,
         Instruction.ROT_L.value, Instruction.HARVEST.value, Instruction.SETTLE.value,
         Instruction.FORWARD.value, Instruction.ROT_R.value]

show(InstructionList(rules, cursor=2, patrol=True))


### 4. Piece extensions

#| export
_PIECE_RENDERERS = {
    PieceType.PAWN:   'pawnPiece',
    PieceType.KNIGHT: 'knightPiece',
    PieceType.ROOK:   'rookPiece',
    PieceType.BISHOP: 'bishopPiece',
    PieceType.QUEEN:  'queenPiece',
    PieceType.KING:   'kingPiece',
}

_PIECE_ICONS = {
    PieceType.PAWN:   "♟", PieceType.KNIGHT: "♞", PieceType.ROOK:   "♜",
    PieceType.BISHOP: "♝", PieceType.QUEEN:  "♛", PieceType.KING:   "♚",
}


@patch
def _ensure_pattern(self: Piece, grid: HexGrid, pat_scale: float = 0.1) -> str:
    """Register flag pattern on grid builder, return CSS fill reference.
    Falls back to flag.primary solid color if no flag."""
    if not self.flag: return "#888"
    pat_name = f"piece_pat_{self.id[:8]}"
    pat = self.flag.flagPattern(pat_name, scale=pat_scale)
    grid.builder.add_definition(pat)
    return f"url(#{pat_name})"


@patch
def draw_svg(self: Piece, grid: HexGrid, hex_idx: int = None,
             scale: float = None, opacity: float = 1.0,
             pat_fill: str = None, suffix: str = "") -> str:
    """Render this piece's chess glyph as SVG at a hex position.

    Args:
        grid:     HexGrid to draw on (needed for hex centres + builder)
        hex_idx:  fine grid index to draw at (defaults to self.location)
        scale:    piece scale (defaults to ~75% of hex diameter)
        opacity:  1.0 for solid, 0.4 for ghost, etc.
        pat_fill: override the fill (otherwise registers flag pattern)
        suffix:   appended to piece_id for uniqueness (e.g. '_ghost')
    """
    idx = hex_idx if hex_idx is not None else self.location
    if idx is None or idx < 0 or idx >= len(grid.hexes): return ""

    center = grid.hexes[idx].center
    renderer = _PIECE_RENDERERS.get(self.piece_type, 'pawnPiece')
    if not self.flag: return ""

    if scale is None:
        scale = (grid.radius * 0.75) / 22.5  # chess SVGs ~22.5px baseline

    fill = pat_fill or self._ensure_pattern(grid)
    pid  = f"piece_{self.id[:8]}{suffix}"

    svg = getattr(self.flag, renderer)(
        MapCord(center.x, center.y),
        scale=scale,
        piece_id=pid,
        fill=fill,
    )

    # Wrap in a group with opacity if not fully opaque
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>\n'

    return svg


@patch
def draw_ghost(self: Piece, grid: HexGrid, hex_idx: int,
               scale: float = None, opacity: float = 0.4) -> str:
    """Draw a faded copy at a projected future position."""
    s = scale or (grid.radius * 0.75) / 22.5
    return self.draw_svg(grid, hex_idx=hex_idx,
                         scale=s * 0.7, opacity=opacity,
                         suffix="_ghost")





In [ ]:
@patch
def draw_svg(self: Piece, grid: HexGrid, hex_idx: int = None,
             scale: float = None, opacity: float = 1.0,
             suffix: str = "") -> str:
    idx = hex_idx if hex_idx is not None else self.location
    if idx is None or idx < 0 or idx >= len(grid.hexes) or not self.flag:
        return ""

    center = grid.hexes[idx].center
    if scale is None:
        scale = (grid.radius * 0.75) / 22.5

    # Auto-select tier from scale
    size = 'board' if scale < 0.6 else ('list' if scale < 1.5 else 'large')

    return self.flag.draw_piece(
        self.piece_type, center, grid.builder,
        scale=scale, size=size,
        piece_id=f"piece_{self.id[:8]}{suffix}",
        opacity=opacity,
    )


@patch
def draw_ghost(self: Piece, grid: HexGrid, hex_idx: int,
               scale: float = None, opacity: float = 0.4) -> str:
    s = scale or (grid.radius * 0.75) / 22.5
    return self.draw_svg(grid, hex_idx=hex_idx,
                         scale=s * 0.7, opacity=opacity,
                         suffix="_ghost")


So we are going to need icons at different sizes. the larger

In [ ]:
#| export
def StatBar(label, current, maximum, color="auto"):
    """A labeled progress bar for bounded stats like health/food."""
    pct = min(100, max(0, (current / maximum * 100))) if maximum else 0
    if color == "auto":
        color = "success" if pct > 60 else "warning" if pct > 30 else "error"
    return Div(
        DivFullySpaced(
            P(label, cls=TextPresets.muted_sm),
            P(f"{int(current)}/{int(maximum)}", cls="text-xs font-mono"),
        ),
        Progress(value=str(int(pct)), max="100",
                 cls=f"uk-progress h-2 [&::-webkit-progress-value]:bg-{color}"),
        cls="space-y-0.5",
    )

@patch
def __ft__(self: Piece):
    """FastHTML card component for piece detail panel."""
    icon = _PIECE_ICONS.get(self.piece_type, "?")
    ptype = self.piece_type.name.title()
    short_id = self.id[:12] + "…"
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # Progress bars for depletable stats
    bars = [
        StatBar("Health", self.health, self.max_health),
        StatBar("Food", self.food, self.food_capacity),
    ]

    # Numeric stats
    num_stats = [
        ("⚔ Attack",  self.attack_strength),
        ("🏃 Speed",   self.move_strength),
        ("🌾 Harvest", self.harvest_strength),
        ("👁 Sight",   self.sight),
    ]
    stat_grid = Grid(
        *[Div(
            P(label, cls=TextPresets.muted_sm),
            P(str(val), cls="font-bold text-center"),
            cls="text-center",
        ) for label, val in num_stats],
        cols=4,
    )

    return Card(
        CardHeader(
            DivFullySpaced(
                DivLAligned(
                    P(icon, cls="text-3xl"),
                    Div(
                        H4(ptype, cls="font-bold"),
                        P(short_id, cls=TextPresets.muted_sm),
                    ),
                ),
                P(f"Facing {facing_arrow}", cls="text-lg"),
            ),
        ),
        CardBody(
            Div(*bars, cls="space-y-2"),
            Divider(),
            stat_grid,
            Divider(),
            InstructionList(self.rules, self.cursor, self.patrol),
        ),
        CardFooter(
            DivFullySpaced(
                Button("Edit Rules",
                       hx_get=f"/piece_rules/{self.id}",
                       hx_target="#right-panel",
                       cls=ButtonT.primary + " " + ButtonT.sm),
                Button("Show Plan",
                       hx_get=f"/piece_plan/{self.id}",
                       hx_target="#map",
                       cls=ButtonT.sm),
            ),
        ),
        id=f"piece-card-{self.id[:8]}",
    )


@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_rounds: int = 10, min_elev: float = 1.0,
             countries: np.ndarray = None) -> list['PieceStep']:
    """Walk rules on coarse grid for N steps."""
    if not self.rules or self.location is None: return []
    steps = []
    pos, fac, cur = self.location, self.facing, self.cursor

    for n in range(num_rounds):
        if cur >= len(self.rules):
            if self.patrol: cur = 0
            else: break

        instr = Instruction(self.rules[cur])
        prev, blocked = pos, False

        if instr == Instruction.FORWARD:
            d = HexPosition.directions()[fac % 6]
            nbr = grid.hexposition_to_index(d, pos)
            passable = (0 <= nbr < len(elevations)
                        and nbr not in grid.invalidRegion
                        and elevations[nbr] >= min_elev)
            # Also block if country field marks it as water/mountain (-1)
            if passable and countries is not None:
                passable = countries[nbr] >= 0
            if passable:
                pos = nbr
            else:
                blocked = True
        elif instr == Instruction.ROT_L:
            fac = (fac - 1) % 6
        elif instr == Instruction.ROT_R:
            fac = (fac + 1) % 6

        steps.append(PieceStep(hex_idx=pos, facing=fac, instruction=instr,
                               prev_idx=prev, step=n, blocked=blocked))
        cur += 1
    return steps



@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_turns: int = 10,
             countries: np.ndarray = None,
             occupied: set[int] = None) -> list['PieceStep']:
    """Simulate piece over multiple turns using move_strength as per-turn budget.

    Budget rules:
      - FORWARD: costs _move_cost (1 flat, 2 uphill). If can't afford → end turn.
      - ROT_L / ROT_R: free
      - PAUSE / DEFEND / HARVEST / GIVE / SETTLE: cost 1
      - FORWARD into water/edge: auto-rotate left (free, up to 5×). Full circle → stuck, cost 1.
      - FORWARD into occupied hex: pause instead, cost 1.

    End-of-rules:
      - Last instruction was non-movement → pause out remaining budget.
      - Last instruction was movement/rotate → wrap cursor to 0.
    """
    if not self.rules or self.location is None:
        return []

    occupied = set(occupied) if occupied else set()
    steps = []
    pos, fac, cur = self.location, self.facing, self.cursor
    step_n = 0

    for turn in range(num_turns):
        budget = float(self.move_strength)

        while budget > 0:
            # --- End of rules ---
            if cur >= len(self.rules):
                last = Instruction(self.rules[-1])
                if last in _COSTS_ONE:
                    # Idle out the turn
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE,
                                           pos, step_n, turn, budget, 0.0))
                    step_n += 1
                    break
                cur = 0  # wrap around

            instr = Instruction(self.rules[cur])
            prev = pos

            # --- FREE: rotations ---
            if instr in _FREE:
                fac = ((fac - 1) if instr == Instruction.ROT_L
                       else (fac + 1)) % 6
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 0.0, budget))
                step_n += 1; cur += 1
                continue

            # --- COST 1: non-movement actions ---
            if instr in _COSTS_ONE:
                budget -= 1.0
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 1.0, budget))
                step_n += 1; cur += 1
                continue

            # --- FORWARD ---
            auto_rots = 0
            moved = False
            saved_fac = fac

            while auto_rots <= 5:
                d = HexPosition.directions()[fac % 6]
                nbr = grid.hexposition_to_index(d, pos)

                passable = (0 <= nbr < len(elevations)
                            and nbr not in grid.invalidRegion
                            and elevations[nbr] >= 1)
                if passable and countries is not None:
                    passable = countries[nbr] >= 0

                if not passable:
                    fac = (fac - 1) % 6   # free auto-rotate left
                    auto_rots += 1
                    continue

                # Occupied → forced pause
                if nbr in occupied:
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                           step_n, turn, 1.0, budget - 1.0,
                                           blocked=True,
                                           auto_rotations=auto_rots))
                    budget -= 1.0
                    step_n += 1; cur += 1
                    moved = True
                    break

                # Can we afford it?
                cost = _move_cost(elevations, pos, nbr)
                if cost > budget:
                    budget = 0       # turn over, can't afford
                    moved = True
                    break

                # Move
                occupied.discard(pos)
                pos = nbr
                occupied.add(pos)
                budget -= cost
                steps.append(PieceStep(pos, fac, Instruction.FORWARD, prev,
                                       step_n, turn, cost, budget,
                                       auto_rotations=auto_rots))
                step_n += 1; cur += 1
                moved = True
                break

            if not moved:
                # Spun full circle — stuck
                fac = saved_fac      # 6 left rotations = back to original
                steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                       step_n, turn, 1.0, budget - 1.0,
                                       blocked=True, auto_rotations=6))
                budget -= 1.0
                step_n += 1; cur += 1

    return steps


def steps_by_turn(steps: list[PieceStep]) -> dict[int, list[PieceStep]]:
    """Group steps by turn number."""
    turns = {}
    for s in steps:
        turns.setdefault(s.turn, []).append(s)
    return turns


@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_turns: int = 10, padding: int = 3,
                countries: np.ndarray = None,
                occupied: set[int] = None
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_turns,
                          countries=countries, occupied=occupied)
    visited = {self.location}
    for s in steps:
        visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations):
                expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)


#| export
def steps_by_turn(steps: list[PieceStep]) -> dict[int, list[PieceStep]]:
    """Group steps by turn number."""
    turns = {}
    for s in steps:
        turns.setdefault(s.turn, []).append(s)
    return turns


@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_turns: int = 10, padding: int = 3,
                countries: np.ndarray = None,
                occupied: set[int] = None
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_turns,
                          countries=countries, occupied=occupied)
    visited = {self.location}
    for s in steps:
        visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations):
                expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

No, they're quite different. Here's a breakdown:

**Note block version** (older/simpler):
- No `turn`/budget concept — just a flat loop over `num_rounds` steps
- One instruction = one step, no cost accounting
- Simple `blocked` bool, no `auto_rotations`
- `PieceStep` constructed with positional args missing `turn`, `cost`, `budget_remaining`
- Handles `patrol` wrap but no end-of-rules idle logic
- No `occupied` set tracking

**Code block version** (newer/full budget model):
- Implements the full **budget model** from your notes — `move_strength` as per-turn spend
- `FORWARD` costs 1 (flat) or 2 (uphill), other actions cost 1, rotations are free
- Tracks `auto_rotations` (up to 5 free spins before stuck-pause)
- Handles occupied hex → forced pause
- End-of-rules: idles if last was non-movement, wraps if movement/rotation
- `PieceStep` gets full `turn`, `cost`, `budget_remaining`, `auto_rotations` fields
- Updates `occupied` set as the piece moves

The **code block version supersedes the note block version** — the note block one is an old stub that predates the budget model design. You can safely ignore/delete the note block version.

One subtle issue: the code block version also **redefines** `simulate` twice (once in `_d55d2b5b`, once in `_743eb4cf`), and those two *are* identical — the second one is just a copy/paste duplicate.

## Build our testing envoirnment.

### 1. Piece Board

In [ ]:
#| export
@dataclass
class PieceBoard:
    """Lightweight playground for testing piece movement and overlays."""
    grid: HexGrid
    elevations: np.ndarray
    pieces: list  # list[Piece]
    countries: np.ndarray = None
    flags: dict = None  # countryId -> CountryFlag

    @classmethod
    def flat(cls, rings=5, radius=20, num_hexes_elev=200):
        """Open grassland — uniform elevation, no obstacles."""
        style = StyleCSS("pb_hex", fill="#e8e8e0", stroke="#ccc", stroke_width=0.5)
        grid = HexGrid.centered(rings=rings, radius=radius, style=style,
                                center=MapCord(radius*(2*rings+2), radius*(2*rings+2)))
        n = len(grid.hexes)
        elevations = np.full(n, num_hexes_elev, dtype=float)
        countries = np.zeros(n, dtype=float)
        # Mark invalid hexes as water
        for idx in grid.invalidRegion:
            elevations[idx] = 0
            countries[idx] = -1
        return cls(grid=grid, elevations=elevations, pieces=[],
                   countries=countries, flags={})

    @classmethod
    def hilly(cls, rings=5, radius=20, seed=42):
        """Rolling terrain with a river of water hexes through the middle."""
        board = cls.flat(rings=rings, radius=radius)
        rng = np.random.default_rng(seed)
        grid = board.grid
        n = len(grid.hexes)
        # Gentle elevation gradient + noise
        for i in range(n):
            if i in grid.invalidRegion: continue
            pos = grid.index_to_hexposition(i)
            board.elevations[i] = 150 + pos.q * 30 + rng.normal(0, 40)
            board.elevations[i] = max(1, board.elevations[i])
        # Carve a river: zero-elev stripe along r=0
        for i in range(n):
            if i in grid.invalidRegion: continue
            pos = grid.index_to_hexposition(i)
            if abs(pos.r) <= 0 and abs(pos.q) <= rings:
                board.elevations[i] = 0
                board.countries[i] = -1
        return board


In [ ]:
CountryFlag.commonNames??

In [ ]:
#| export
@patch
def add_piece(self: PieceBoard, hex_idx: int, piece_type=PieceType.PAWN,
              facing: int = 0, instructions: InstructionList = None,
              country_id: int = 1, birth_year: int = 1987) -> Piece:
    if instructions is None:
        instructions = InstructionList()
    if country_id not in self.flags:
        palette = CountryFlag.seaborn("husl", levels=max(country_id + 2, 4))
        self.flags[country_id] = palette[country_id]

    try:
        names = (CountryFlag.commonNames(year=birth_year, gender="M") +
                 CountryFlag.commonNames(year=birth_year, gender="F"))
        name = random.choice(names)
    except (ValueError, IndexError):
        name = "Marvin"

    piece = Piece(
        location=hex_idx, piece_type=piece_type,
        facing=facing,
        instructions=instructions,
        owner_id=country_id,
        name=name,
        birth_year=birth_year,
    )
    piece.flag = self.flags[country_id]
    self.pieces.append(piece)
    if self.countries is not None and self.countries[hex_idx] == 0:
        self.countries[hex_idx] = country_id
    return piece


In [ ]:
#| export
# ── Elevation palette for the playground ─────────────────────────────────────
_PB_WATER  = StyleCSS("pb_water",  fill="#6baed6", stroke="#4292c6", stroke_width=0.5)
_PB_LEVELS = [
    StyleCSS("pb_e0", fill="#c7e9c0", stroke="#bbb", stroke_width=0.5),  # low
    StyleCSS("pb_e1", fill="#a1d99b", stroke="#bbb", stroke_width=0.5),
    StyleCSS("pb_e2", fill="#74c476", stroke="#bbb", stroke_width=0.5),
    StyleCSS("pb_e3", fill="#d4b483", stroke="#bbb", stroke_width=0.5),  # hill
    StyleCSS("pb_e4", fill="#bc8a5f", stroke="#bbb", stroke_width=0.5),
    StyleCSS("pb_e5", fill="#9e6b4a", stroke="#999", stroke_width=0.5),  # peak
]

def _elev_style(elev: float, delta: float = 80.0) -> StyleCSS:
    """Map elevation to a terrain style."""
    if elev < 1: return _PB_WATER
    level = min(len(_PB_LEVELS) - 1, int(elev / delta))
    return _PB_LEVELS[level]


@patch
def _apply_terrain(self: PieceBoard):
    """Paint each hex with its elevation style, registering styles on the builder."""
    grid = self.grid
    for s in [_PB_WATER] + _PB_LEVELS:
        grid.builder.add_style(s)
    for i, hex_obj in enumerate(grid.hexes):
        if i in grid.invalidRegion: continue
        hex_obj.style = _elev_style(self.elevations[i])


@patch
def _pieces_overlay(self: PieceBoard) -> str:
    """SVG string: all pieces at their current locations."""
    svg = ""
    for p in self.pieces:
        svg += p.draw_svg(self.grid)
    return svg






#| export
@patch
def _plan_overlay(self: PieceBoard, num_rounds: int = 12) -> str:
    """SVG string: simulated movement paths + action glyphs for all pieces."""
    svg = ""
    grid = self.grid

    for p in self.pieces:
        if not p.rules or p.location is None: continue
        steps = p.simulate(grid, self.elevations,
                           num_rounds=num_rounds,
                           countries=self.countries)
        if not steps: continue

        color = p.flag.primary if p.flag else "#333"
        g     = DiagramGlyphs(color=color, size=grid.radius * 0.45)

        # Start ring around piece's current position
        origin = grid.hexes[p.location].center
        svg += g.start_marker(origin.x, origin.y, r=grid.radius * 0.55)

        # Arrow style — dashed, colour-matched to piece
        arrow_style = StyleCSS(
            f"plan_arrow_{p.id[:8]}",
            stroke=color, stroke_width=max(1, grid.radius * 0.08),
            fill="none", opacity="0.7",
            stroke_dasharray=f"{grid.radius*0.25:.0f},{grid.radius*0.12:.0f}",
        )
        grid.builder.add_style(arrow_style)

        prev_idx = p.location
        for step in steps:
            cur = grid.hexes[step.hex_idx].center

            if step.instruction == Instruction.FORWARD and not step.blocked:
                if prev_idx != step.hex_idx:
                    svg += grid.arrow(prev_idx, step.hex_idx,
                                      style=arrow_style, factor=0.3)
                svg += p.draw_ghost(grid, step.hex_idx)

            elif step.blocked:
                svg += g.blocked_x(cur.x, cur.y)

            else:
                svg += g.action_glyph(step.instruction, cur.x, cur.y)

            svg += g.step_label(cur.x, cur.y, step.step)
            prev_idx = step.hex_idx

    return svg


#| export
@patch
def render(self: PieceBoard, show_plan: bool = False,
           num_rounds: int = 12, width: int = None, height: int = None) -> str:
    """Return a full SVG string of the board."""
    self._apply_terrain()
    grid = self.grid
    grid.update()

    grid.builder.adjust("pieces", self._pieces_overlay())
    grid.builder.adjust("plan", self._plan_overlay(num_rounds) if show_plan else "")

    if width is None or height is None:
        # Derive bounding box from actual hex geometry
        xs = [h.center.x for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        ys = [h.center.y for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        pad = grid.radius * 1.5
        w = int(max(xs) + pad) if width  is None else width
        h = int(max(ys) + pad) if height is None else height
    else:
        w, h = width, height

    grid.builder.width  = w
    grid.builder.height = h
    return grid.builder.xml()


#| export
@patch
def __ft__(self: PieceBoard):
    """Default view — terrain + pieces, no plan."""
    return Div(NotStr(self.render()))


def PieceBoardPlan(board: PieceBoard, num_rounds: int = 12):
    """View with movement plan overlay."""
    return Div(NotStr(board.render(show_plan=True, num_rounds=num_rounds)))


#### swapping up

In [ ]:
#| export
def _turn_end_labels(steps: list[PieceStep]) -> list[tuple[int, int, int]]:
    """Merge consecutive turns that end on the same hex into ranges.
    
    Returns [(hex_idx, first_turn, last_turn), ...] — one entry per
    contiguous run of turns ending at the same hex.
    """
    # Find the hex where each turn ends
    turn_hex = {}  # turn -> hex_idx
    for s in steps:
        turn_hex[s.turn] = s.hex_idx

    if not turn_hex:
        return []

    sorted_turns = sorted(turn_hex)
    runs = []
    run_start = sorted_turns[0]
    prev_turn = run_start
    prev_hex  = turn_hex[run_start]

    for t in sorted_turns[1:]:
        if turn_hex[t] == prev_hex and t == prev_turn + 1:
            prev_turn = t  # extend the run
        else:
            runs.append((prev_hex, run_start, prev_turn))
            run_start = t
            prev_turn = t
            prev_hex  = turn_hex[t]
    runs.append((prev_hex, run_start, prev_turn))
    return runs


In [ ]:
#| export
def _range_label(g: DiagramGlyphs, cx, cy, t0, t1, color):
    """Emit '3' for a single turn or '2–4' for a range (1-indexed)."""
    text = str(t0 + 1) if t0 == t1 else f"{t0+1}–{t1+1}"
    offset    = g.size * 0.35
    font_size = max(12, g.size * 0.3)
    return (f'<text x="{cx+offset:.1f}" y="{cy-offset:.1f}" '
            f'text-anchor="start" font-size="{font_size:.0f}" '
            f'fill="{color}" font-family="sans-serif" '
            f'font-weight="bold">{text}</text>\n')


In [ ]:
#| export
def piece_plan_overlay(piece: Piece, grid: HexGrid,
                       steps: list[PieceStep] = None,
                       c2f: dict = None,
                       num_turns: int = 12,
                       elevations: np.ndarray = None,
                       countries: np.ndarray = None) -> str:
    if steps is None:
        if elevations is None: return ""
        steps = piece.simulate(grid, elevations, num_turns=num_turns,
                               countries=countries)
    if not steps: return ""

    svg, N = "", len(grid.hexes)
    r = grid.radius
    color = piece.flag.primary if piece.flag else "#333"
    comp  = piece.flag.comp    if piece.flag else "#555"
    g = DiagramGlyphs(color=color, size=r * 0.45)

    arrow_style = StyleCSS(
        f"plan_arr_{piece.id[:8]}",
        stroke=color, stroke_width=max(1, r * 0.08),
        fill="none", opacity="0.7",
        stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}",
    )
    grid.builder.add_style(arrow_style)

    # ── Find last step of each turn ──
    turn_ends = {}
    for i, s in enumerate(steps):
        turn_ends[s.turn] = i  # last one wins

    # ── Start marker ──
    start = _map_point(piece.location, c2f)
    if 0 <= start < N:
        svg += piece.draw_svg(grid, hex_idx=start)
        c = grid.hexes[start].center
        svg += g.start_marker(c.x, c.y, r=r * 0.55)

    # ── Each step ──
    prev_idx = start
    for i, step in enumerate(steps):
        idx = _map_point(step.hex_idx, c2f)
        if idx < 0 or idx >= N:
            prev_idx = idx
            continue
        cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y

        if step.instruction == Instruction.FORWARD and not step.blocked:
            if 0 <= prev_idx < N and prev_idx != idx:
                svg += grid.arrow(prev_idx, idx, style=arrow_style, factor=0.3)
            svg += piece.draw_ghost(grid, idx)
        elif step.blocked:
            svg += g.blocked_x(cx, cy)
        else:
            svg += g.action_glyph(step.instruction, cx, cy)

        # Only label the last step of each turn
        #if i == turn_ends[step.turn]:
        #    svg += g.step_label(cx, cy, step.turn, color=comp)

        prev_idx = idx

    # ── Range labels (after the step loop) ──
    for hex_idx, t0, t1 in _turn_end_labels(steps):
        idx = _map_point(hex_idx, c2f)
        if 0 <= idx < N:
            cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y
            svg += _range_label(g, cx, cy, t0, t1, comp)


    return svg


In [ ]:
#| export
@patch
def _plan_overlay(self: PieceBoard, num_turns: int = 12) -> str:
    """SVG string: simulated movement plan for all pieces."""
    svg = ""
    for p in self.pieces:
        svg += piece_plan_overlay(
            p, self.grid, num_turns=num_turns,
            elevations=self.elevations, countries=self.countries,
        )
    return svg


In [ ]:
#| export
@patch
def render(self: PieceBoard, show_plan: bool = False,
           num_turns: int = 12, width: int = None, height: int = None) -> str:
    """Return a full SVG string of the board."""
    self._apply_terrain()
    grid = self.grid
    grid.update()

    grid.builder.adjust("pieces", self._pieces_overlay())
    grid.builder.adjust("plan", self._plan_overlay(num_turns) if show_plan else "")

    if width is None or height is None:
        xs = [h.center.x for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        ys = [h.center.y for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        pad = grid.radius * 1.5
        w = int(max(xs) + pad) if width  is None else width
        h = int(max(ys) + pad) if height is None else height
    else:
        w, h = width, height

    grid.builder.width  = w
    grid.builder.height = h
    return grid.builder.xml()


In [ ]:
#| export
@patch
def __ft__(self: PieceBoard):
    """Default view — terrain + pieces, no plan."""
    return Div(NotStr(self.render()))


def PieceBoardPlan(board: PieceBoard, num_turns: int = 12):
    """View with movement plan overlay."""
    return Div(NotStr(board.render(show_plan=True, num_turns=num_turns)))


In [ ]:
??InstructionList

In [ ]:
#| export
# ── Quick PieceBoard demo ─────────────────────────────────────────────────────
board = PieceBoard.hilly(rings=5, radius=22, seed=7)
grid  = board.grid
mid   = grid.middle

scout = board.add_piece(
    mid, piece_type=PieceType.KNIGHT, facing=3, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
    ]),
)

worker = board.add_piece(
    mid - 3, piece_type=PieceType.PAWN, facing=4, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.HARVEST.value, Instruction.HARVEST.value,
        Instruction.ROT_L.value, Instruction.ROT_L.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.GIVE.value,
    ]),
)

guard = board.add_piece(
    mid + 7, piece_type=PieceType.ROOK, facing=0, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.DEFEND.value, Instruction.DEFEND.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value,
    ]),
)

bishop = board.add_piece(
    mid + grid.nCols * 2, piece_type=PieceType.BISHOP, facing=5, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value,
        Instruction.PAUSE.value,
        Instruction.SETTLE.value,
    ]),
)

PieceBoardPlan(board, num_turns=3)


Anything else you would change?A few things I'd tighten up, roughly in priority order:

**1. Ghost IDs aren't unique** — every ghost gets `id="piece_66ff270b_ghost"`, so you end up with duplicate IDs in the SVG. Some browsers get confused. Quick fix in `draw_ghost` / `draw_svg`:

```python
suffix=f"_ghost_{step_n}"  # or pass step index through
```

**2. Pattern defs are registered once per draw call** — look at the output: `piece_pat_66ff270b` appears as a `<pattern>` 15+ times. `_ensure_pattern` creates a new object each call; `add_definition` deduplicates by `id` but still runs every time. A one-liner cache would help:

```python
@patch
def _ensure_pattern(self: Piece, grid: HexGrid, pat_scale: float = 0.1) -> str:
    if not self.flag: return "#888"
    pat_name = f"piece_pat_{self.id[:8]}"
    if pat_name not in grid.builder.styles:  # or track in a set
        pat = self.flag.flagPattern(pat_name, scale=pat_scale)
        grid.builder.add_definition(pat)
    return f"url(#{pat_name})"
```

**3. `occupied` isn't threaded between pieces** — `_plan_overlay` calls `piece_plan_overlay` per piece but doesn't accumulate an `occupied` set across them. So two pieces can happily walk through each other in the preview. If you want collision-aware plans, build the set once and pass it through in chess-rank order.

**4. Budget can go negative** — in the "blocked" and "stuck" branches, `budget -= 1.0` fires without checking `budget >= 1.0`. If a piece has 0.5 budget left and gets blocked, `budget_remaining` becomes –0.5. A `budget = max(0, budget - 1.0)` guard would be cleaner.

**5. Food/diet isn't deducted yet** — the design says deduct `diet` at turn end and lose health when `food < 0`, but `simulate` doesn't touch `food` or `health`. Not a bug today since it's all read-only simulation, but worth a `# TODO` so you remember when you wire up actual round resolution.

The first two are the most immediately impactful — they affect every render right now.

In [ ]:
@rt
def hello(name: str): return P(f"Hello {name}!")

webMe(Div(
    H3("Say Hello"),
    Form(
        Input(placeholder="Your name...", id='name'),
        Button("Send"),
        hx_get=hello, hx_target="#result"
    ),
    Div(id='result'),
))

#| export
@patch
def create_game(self: GameStorage, user_id, template_name="bayArea",
                kingdoms=5, lakes=1, radius=10) -> ActiveGame:
    logging.info(f"create_game: user={user_id} template={template_name}")
    tt = TerrainTemplate()
    terrain = getattr(tt, template_name)()
    
    terrain.carve_to_ocean(num_lakes=lakes)
    terrain.hexGrid.adjustRadius(radius)

    # GameBoard now creates cover + basins internally
    board = GameBoard(terrain, top_n=kingdoms)
    board.expand_kingdoms(max_rounds=50)
    
    # Save using the cover that GameBoard created
    board.cover.db = self
    board.cover.save(name=template_name)
    board.save(self, world_id=board.cover.ident)

    self.db.execute("UPDATE user SET activeWorld = ? WHERE id = ?",
                    [board.cover.ident, user_id])
    
    return ActiveGame(board=board, cover=board.cover, world_id=board.cover.ident)


_game_cache = {}
_cache_lock = threading.Lock()

@patch
def active_board(self: GameStorage, user_id) -> ActiveGame:
    with _cache_lock:
        if user_id in _game_cache:
            logging.info(f"active_board: cache hit for user {user_id}")
            return _game_cache[user_id]
    
    logging.info(f"active_board: cache miss, building for user {user_id}")
    row = self.db.execute("SELECT activeWorld FROM user WHERE id = ?", [user_id]).fetchone()
    if not row or not row[0]:
        return None
    
    world_id = row[0]
    try:
        # gameboard() now loads cover internally and reuses cover.basin
        board = self.gameboard(world_id)
        active = ActiveGame(board=board, cover=board.cover, world_id=world_id)
        
        with _cache_lock:
            _game_cache[user_id] = active
        
        return active
    except Exception as e:
        logging.error(f"active_board: FAILED: {e}", exc_info=True)
        return None


def piece_plan_overlay(piece: Piece, grid: HexGrid,
                       steps: list[PieceStep] = None,
                       c2f: dict = None,
                       num_turns: int = 12,
                       elevations: np.ndarray = None,
                       countries: np.ndarray = None) -> str:
    """Render movement plan overlay for a single piece.

    Pass pre-computed `steps`, or provide `elevations` to simulate on the fly.
    """
    if steps is None:
        if elevations is None:
            return ""
        steps = piece.simulate(grid, elevations, num_turns=num_turns,
                               countries=countries)
    if not steps:
        return ""

    svg, N = "", len(grid.hexes)
    r = grid.radius
    color = piece.flag.primary if piece.flag else "#333"
    comp  = piece.flag.comp    if piece.flag else "#555"
    g = DiagramGlyphs(color=color, size=r * 0.45)

    # Dashed arrow style for movement lines
    arrow_style = StyleCSS(
        f"plan_arr_{piece.id[:8]}",
        stroke=color, stroke_width=max(1, r * 0.08),
        fill="none", opacity="0.7",
        stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}",
    )
    grid.builder.add_style(arrow_style)

    # ── Start: full piece + ring ──
    start = _map_point(piece.location, c2f)
    if 0 <= start < N:
        svg += piece.draw_svg(grid, hex_idx=start)
        c = grid.hexes[start].center
        svg += g.start_marker(c.x, c.y, r=r * 0.55)

    # ── Each step ──
    prev_idx = start
    for step in steps:
        idx = _map_point(step.hex_idx, c2f)
        if idx < 0 or idx >= N:
            prev_idx = idx
            continue
        cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y

        if step.instruction == Instruction.FORWARD and not step.blocked:
            if 0 <= prev_idx < N and prev_idx != idx:
                svg += grid.arrow(prev_idx, idx, style=arrow_style, factor=0.3)
            svg += piece.draw_ghost(grid, idx)

        elif step.blocked:
            svg += g.blocked_x(cx, cy)

        else:
            svg += g.action_glyph(step.instruction, cx, cy)

        svg += g.step_label(cx, cy, step.step, color=comp)
        prev_idx = idx

    return svg


#### some junk

@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    logging.info(f"create world redirecting")
    return RedirectResponse('/game', status_code=303)


@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    logging.info(f"create_world: session keys={list(session.keys())}")
    invalidate_cache(uid)  # <-- clear old game
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    return RedirectResponse('/game', status_code=303)

In [ ]:
# Empty before we go

In [ ]:
@rt("/piece_detail/{id}")
def piece_detail(session, id: str):
    """Placeholder for piece detail — right panel."""
    return Div(
        H4(f"🧑 Piece", cls="font-bold"),
        P(f"ID: {id[:12]}…", cls=TextPresets.muted_sm),
        Divider(),
        P("Actions and stats coming soon", cls="opacity-50 text-sm"),
        cls="space-y-2"
    )


## Editors

### 1. Piece Drag editor

In [ ]:
#| export
def piece_drag_editor(piece: Piece, rings: int = 3, radius: int = 35):
    """HexDragMap centered on piece location.
    Drag a path → POST hex_ids → convert to rules → display compact badge strip.
    """
    base    = StyleCSS("pe-hex",   fill="#1e293b", stroke="#475569", stroke_width=1)
    current = StyleCSS("pe-cur",   fill="#f59e0b", stroke="#fbbf24", stroke_width=2)
    lbl     = StyleCSS("pe-label", fill="#e2e8f0", font_size="11px")

    grid = HexGrid.centered(rings=rings, radius=radius, style=base)
    for s in [current, lbl]:
        grid.builder.add_style(s)

    mid = grid.middle

    grid.hexes[mid].style      = current
    grid.hexes[mid].label      = piece.piece_type.value[0].upper()
    grid.hexes[mid].labelStyle = lbl.name

    facing_dir = HexPosition.directions()[piece.facing % 6]
    facing_idx = grid.hexposition_to_index(facing_dir, mid)
    if facing_idx >= 0:
        grid.builder.adjust("facing", grid.arrow(
            mid, facing_idx,
            style=StyleCSS("pe-arrow", stroke="#f59e0b", stroke_width=2),
        ))

    grid.update()

    return Div(
        H5("Drag a patrol path", cls="text-sm font-bold mb-1"),
        P("Drag across hexes to generate movement instructions",
          cls="text-xs opacity-50 mb-2"),
        HexDragMap(
            grid,
            on_drag={
                "hx_post":   f"/piece_path/{piece.id}/{piece.facing}",
                "hx_target": f"#rules-{piece.id[:8]}",
                "hx_swap":   "outerHTML",
            },
            id=f"drag-editor-{piece.id[:8]}",
        ),
        piece.instructions.compact(piece.id),
        cls="space-y-2",
    )


In [ ]:
#| export
@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request: Request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, persist, return compact badge strip."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # Reconstruct the same grid used in piece_drag_editor
    rings, radius = 3, 35
    grid = HexGrid.centered(rings=rings, radius=radius,
                            style=StyleCSS("x", fill="white"))

    new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)

    # Persist — append to existing rules
    try:
        rec = globalStore.pieces[piece_id]
        if rec:
            existing = [int(x) for x in (rec.rules or "").split(",") if x]
            combined = existing + new_rules
            globalStore.pieces.upsert(
                {"id": piece_id, "rules": ",".join(str(r) for r in combined)},
                pk="id",
            )
            new_rules = combined
    except Exception:
        pass  # demo mode — show without saving

    return InstructionList(new_rules).compact(piece_id)


I think these both (piece_path, piece_drag_editor) need to be refactored since we switched piece from having a list of rules to an InstructionList

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're right — both still reference `piece.rules` / `piece.cursor` / `piece.patrol` as flat attributes, but `Piece` now stores `instructions: InstructionList`. Here are the updated versions:

```python
def piece_drag_editor(piece: Piece, rings: int = 3, radius: int = 35):
    """HexDragMap centered on piece location.
    Drag a path → POST hex_ids → convert to rules → display compact badge strip.
    """
    base    = StyleCSS("pe-hex",   fill="#1e293b", stroke="#475569", stroke_width=1)
    current = StyleCSS("pe-cur",   fill="#f59e0b", stroke="#fbbf24", stroke_width=2)
    lbl     = StyleCSS("pe-label", fill="#e2e8f0", font_size="11px")

    grid = HexGrid.centered(rings=rings, radius=radius, style=base)
    for s in [current, lbl]:
        grid.builder.add_style(s)

    mid = grid.middle

    grid.hexes[mid].style      = current
    grid.hexes[mid].label      = piece.piece_type.value[0].upper()
    grid.hexes[mid].labelStyle = lbl.name

    facing_dir = HexPosition.directions()[piece.facing % 6]
    facing_idx = grid.hexposition_to_index(facing_dir, mid)
    if facing_idx >= 0:
        grid.builder.adjust("facing", grid.arrow(
            mid, facing_idx,
            style=StyleCSS("pe-arrow", stroke="#f59e0b", stroke_width=2),
        ))

    grid.update()

    return Div(
        H5("Drag a patrol path", cls="text-sm font-bold mb-1"),
        P("Drag across hexes to generate movement instructions",
          cls="text-xs opacity-50 mb-2"),
        HexDragMap(
            grid,
            on_drag={
                "hx_post":   f"/piece_path/{piece.id}/{piece.facing}",
                "hx_target": f"#rules-{piece.id[:8]}",
                "hx_swap":   "outerHTML",
            },
            id=f"drag-editor-{piece.id[:8]}",
        ),
        piece.instructions.compact(piece.id),
        cls="space-y-2",
    )
```

```python
@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request: Request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, persist, return compact badge strip."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # Reconstruct the same grid used in piece_drag_editor
    rings, radius = 3, 35
    grid = HexGrid.centered(rings=rings, radius=radius,
                            style=StyleCSS("x", fill="white"))

    new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)

    # Persist — append to existing rules
    try:
        rec = globalStore.pieces[piece_id]
        if rec:
            existing = [int(x) for x in (rec.rules or "").split(",") if x]
            combined = existing + new_rules
            globalStore.pieces.upsert(
                {"id": piece_id, "rules": ",".join(str(r) for r in combined)},
                pk="id",
            )
            new_rules = combined
    except Exception:
        pass  # demo mode — show without saving

    return InstructionList(new_rules).compact(piece_id)
```

The changes are minimal:

1. **`piece_drag_editor`**: replaced `InstructionList(piece.rules, piece.cursor, piece.patrol).compact(...)` → `piece.instructions.compact(piece.id)` — no need to reconstruct what already exists.

2. **`piece_path`**: changed `rec.get("rules")` → `rec.rules` (since `PieceRecord` is a dataclass, not a dict). The return still constructs a fresh `InstructionList` from the merged rules, which is correct since we're building from the DB string.

And while you're at it, `add_piece` also needs the same fix — that's what caused the `TypeError` in the demo cell:

```python
@patch
def add_piece(self: PieceBoard, hex_idx: int, piece_type=PieceType.PAWN,
              facing: int = 0, rules: list[int] = None,
              country_id: int = 1) -> Piece:
    if country_id not in self.flags:
        palette = CountryFlag.seaborn("husl", levels=max(country_id + 2, 4))
        self.flags[country_id] = palette[country_id]
    piece = Piece(
        location=hex_idx, piece_type=piece_type,
        facing=facing,
        instructions=InstructionList(rules or []),
        owner_id=country_id,
    )
    piece.flag = self.flags[country_id]
    self.pieces.append(piece)
    if self.countries is not None and self.countries[hex_idx] == 0:
        self.countries[hex_idx] = country_id
    return piece
```

That `rules=` → `instructions=InstructionList(...)` swap is the one-liner that unblocks your demo cell.

Do we have enough to make an editable list of instructions? We want to be able to delete and pick an insertion point where we can use the drag extension

Lets do the apply version. We might want to do heavy lifting on the back end to render

Should RuleEditSession use an InstructionList? should we combine classes?

### 2. Rule Edit Session

In [ ]:
#| export
@dataclass
class RuleEditSession:
    """In-memory edit buffer for a piece's rules."""
    piece_id: str
    instructions: InstructionList

_rule_sessions: dict[str, RuleEditSession] = {}

def _open_session(piece: Piece) -> RuleEditSession:
    sess = RuleEditSession(
        piece_id=piece.id,
        instructions=InstructionList(list(piece.rules), piece.cursor, piece.patrol),
    )
    _rule_sessions[piece.id] = sess
    return sess


In [ ]:
#| export
def _auth_piece(session, piece_id: str) -> tuple[RuleEditSession, int] | None:
    """Return (session, user_id) if the user owns this piece, else None.
    
    Checks:
      1. User is logged in (ensure_user)
      2. Piece belongs to the user's active world
      3. Piece's kingdom matches one the user controls
      4. An edit session exists (for mutation routes)
    """
    uid    = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return None

    # Does this piece belong to the active world?
    try:
        rec = globalStore.pieces[piece_id]
    except (KeyError, NotFoundError):
        return None

    r = rec if isinstance(rec, dict) else asdict(rec)
    if r.get("world_id") != active.world_id:
        return None

    # Does the piece's kingdom belong to this user's game?
    piece_kingdom = r.get("kingdom_id", 0)
    valid_kingdoms = {k.countryId for k in active.board.kingdoms}
    if piece_kingdom not in valid_kingdoms:
        return None

    return uid


def _auth_session(session, piece_id: str) -> RuleEditSession | None:
    """Like _auth_piece but also requires an active edit session."""
    uid = _auth_piece(session, piece_id)
    if uid is None:
        return None
    return _rule_sessions.get(piece_id)


_DENIED = P("⛔ Not authorized", cls="text-sm text-red-500")
_EXPIRED = P("Session expired — reopen the editor", cls="text-sm text-red-500")


In [ ]:
#| export
# ── Shared grid factory ──────────────────────────────────────────────────────
def _editor_grid(rings=3, radius=35):
    return HexGrid.centered(
        rings=rings, radius=radius,
        style=StyleCSS("pe-hex", fill="#1e293b", stroke="#475569", stroke_width=1),
    )

def _editor_id(piece_id): return f"rule-editor-{piece_id[:8]}"


# ── Editor panel component ───────────────────────────────────────────────────
def _rules_editor(sess: RuleEditSession) -> FT:
    """Full server-rendered rules editor. Re-rendered on every mutation."""
    eid  = _editor_id(sess.piece_id)
    il   = sess.instructions
    rows = []

    for i, r in enumerate(il.rules):
        instr     = Instruction(r)
        label, _  = INSTR_LABELS.get(instr, (instr.name, "?"))
        is_cursor = (i == il.cursor)

        cursor_btn = Button(
            "▶" if is_cursor else "○",
            hx_post=piece_rule_cursor.to(piece_id=sess.piece_id, idx=i),
            hx_target=f"#{eid}", hx_swap="outerHTML",
            cls=(ButtonT.primary if is_cursor else ButtonT.ghost) + " " + ButtonT.xs,
        )
        delete_btn = Button(
            UkIcon("x", height=12),
            hx_post=piece_rule_delete.to(piece_id=sess.piece_id, idx=i),
            hx_target=f"#{eid}", hx_swap="outerHTML",
            hx_confirm="Delete this instruction?",
            cls=ButtonT.destructive + " " + ButtonT.xs,
        )
        rows.append([cursor_btn, _instr_glyph(instr), label, delete_btn])

    # "Append at end" cursor slot
    at_end = il.cursor >= len(il.rules)
    rows.append([
        Button("▶" if at_end else "○",
               hx_post=piece_rule_cursor.to(piece_id=sess.piece_id, idx=len(il.rules)),
               hx_target=f"#{eid}", hx_swap="outerHTML",
               cls=(ButtonT.primary if at_end else ButtonT.ghost) + " " + ButtonT.xs),
        "", "— end —", "",
    ])

    tbl = TableFromLists(["", "", "Action", ""], rows,
                         cls=(TableT.sm, TableT.striped))

    patrol_toggle = LabelCheckboxX(
        "Patrol (loop)", checked=il.patrol,
        hx_post=piece_rule_patrol.to(piece_id=sess.piece_id),
        hx_target=f"#{eid}", hx_swap="outerHTML",
    )

    # Drag editor inserts at cursor
    grid = _editor_grid()
    grid.update()
    drag_section = Div(
        P(f"Drag to insert at position {il.cursor + 1}",
          cls="text-xs opacity-60 mb-1"),
        HexDragMap(grid, on_drag={
            "hx_post":   piece_path.to(piece_id=sess.piece_id, facing=0),
            "hx_target": f"#{eid}",
            "hx_swap":   "outerHTML",
        }),
    )

    action_row = DivFullySpaced(
        Button("Apply", cls=ButtonT.primary + " " + ButtonT.sm,
               hx_post=piece_rule_apply.to(piece_id=sess.piece_id),
               hx_target=f"#{eid}", hx_swap="outerHTML",
               hx_indicator=f"#{eid}"),
        Button("Cancel", cls=ButtonT.ghost + " " + ButtonT.sm,
               hx_post=piece_rule_cancel.to(piece_id=sess.piece_id),
               hx_target=f"#{eid}", hx_swap="outerHTML"),
    )

    return Card(
        H5("Edit Rules", cls="font-bold"),
        tbl if il.rules else P("No instructions — drag a path below",
                                cls="opacity-50 text-sm"),
        patrol_toggle,
        Divider(),
        drag_section,
        Divider(),
        action_row,
        id=eid, cls="htmx-indicator",
    )




In [ ]:
#| export
@rt
def piece_rules(session, piece_id: str):
    """Open rules editor from current DB state."""
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    piece = globalStore.piece_from_id(piece_id)
    return _rules_editor(_open_session(piece))

@rt
def piece_rule_cursor(session, piece_id: str, idx: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.set_cursor(idx)
    return _rules_editor(sess)

@rt
def piece_rule_delete(session, piece_id: str, idx: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.delete_at(idx)
    return _rules_editor(sess)

@rt
def piece_rule_patrol(session, piece_id: str):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.toggle_patrol()
    return _rules_editor(sess)

@rt
async def piece_path(session, request, piece_id: str, facing: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    form    = await request.form()
    raw     = form.get("hex_ids", "")
    hex_ids = [int(x) for x in raw.split(",") if x.strip()]
    if hex_ids:
        grid = _editor_grid()
        new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)
        sess.instructions.insert_rules(new_rules)
    return _rules_editor(sess)

@rt
def piece_rule_apply(session, piece_id: str):
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    sess = _rule_sessions.pop(piece_id, None)
    if not sess: return _EXPIRED

    il = sess.instructions
    globalStore.pieces.upsert({
        "id": piece_id,
        "rules": ",".join(str(r) for r in il.rules),
        "cursor": il.cursor,
        "patrol": 1 if il.patrol else 0,
    }, pk="id")

    eid = _editor_id(piece_id)
    editor_result = Div(
        P("✓ Rules saved", cls="text-sm text-green-600 font-bold mb-2"),
        il.compact(piece_id),
        id=eid,
    )
    map_update = Div(P("Map updated", cls="text-xs opacity-50"),
                     id="map", hx_swap_oob="innerHTML")
    return editor_result, map_update

@rt
def piece_rule_cancel(session, piece_id: str):
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    sess = _rule_sessions.pop(piece_id, None)
    eid  = _editor_id(piece_id)
    if sess:
        return Div(sess.instructions.compact(piece_id), id=eid)
    return _EXPIRED


So we want to make sure that the user has permission to edit these routes

Should this use The hx-delete attribute in htmx? issues an HTTP DELETE request to a specified URL, typically to remove a resource on the server. It is used in combination with other htmx attributes like hx-target and hx-swap to define how the page content updates after the request. 

and others. We want this to be as friendly in the eco HTMLX and FASTHTML system as possible

### 3. Direction Picker

In [ ]:
#| export
_DIR_ARROWS = ["↙", "←", "↖", "↗", "→", "↘"]   # keep

@rt
def set_facing(session, piece_id: str, facing: int):
    """Update piece facing — works with edit session if open."""
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED

    sess = _rule_sessions.get(piece_id)
    if sess:
        # If editor is open, update session (uncommitted)
        # facing lives on the Piece, not InstructionList — 
        # so we'd need to stash it. For now just re-render compass.
        pass

    facing = facing % 6
    target_id = f"facing-{piece_id[:8]}"

    # Reload piece, update, re-render compass
    piece = globalStore.piece_from_id(piece_id)
    piece.facing = facing
    globalStore.pieces.upsert({"id": piece_id, "facing": facing}, pk="id")

    return facing_grid(piece)


In [ ]:
#| export
def facing_grid(piece: Piece, radius: int = 30):
    """Compass rose: piece glyph in center, arrows pointing to each direction."""
    piece_id = piece.id
    base   = StyleCSS("fc-hex",    fill="#f8f8f8", stroke="#999", stroke_width=1.5)
    active = StyleCSS("fc-active", fill="#4CAF50", stroke="#333", stroke_width=2)
    center = StyleCSS("fc-center", fill="#e0e0e0", stroke="#666", stroke_width=1.5)

    grid = HexGrid.centered(1, radius=radius, style=base)
    for s in [active, center]:
        grid.builder.add_style(s)

    mid = grid.middle

    # Direction hexes
    dir_map = {}
    for fi, d in enumerate(HexPosition.directions()):
        gi = grid.hexposition_to_index(d, mid)
        dir_map[gi] = fi
        grid.hexes[gi].style = active if fi == piece.facing else base
        grid.hexes[gi].label = _DIR_ARROWS[fi]
        grid.hexes[gi].labelStyle = "fc-label"

    grid.hexes[mid].style = center
    grid.builder.add_style(StyleCSS("fc-label", font_size="13px"))
    grid.update()

    # Overlay: piece in center + arrows from center to each neighbor
    overlay = piece.draw_svg(grid, hex_idx=mid, scale=(radius * 0.6) / 22.5)

    arrow_active = StyleCSS("fc-arr-on",  stroke="#4CAF50", stroke_width=2, fill="none")
    arrow_dim    = StyleCSS("fc-arr-off", stroke="#bbb",    stroke_width=1, fill="none")
    grid.builder.add_style(arrow_active)
    grid.builder.add_style(arrow_dim)

    for gi, fi in dir_map.items():
        style = arrow_active if fi == piece.facing else arrow_dim
        overlay += grid.arrow(mid, gi, style=style, factor=0.35)

    grid.builder.adjust("overlay", overlay)

    target_id = f"facing-{piece_id[:8]}"

    def on_click(g, i):
        if i in dir_map:
            return {
                "hx-post":   f"/set_facing/{piece_id}/{dir_map[i]}",
                "hx-target": f"#{target_id}",
                "hx-swap":   "outerHTML",
            }
        return {}

    return Div(
        H5("Facing", cls="font-bold text-sm mb-1"),
        HexTouchMap(grid, on_click=on_click),
        id=target_id,
        cls="inline-block",
    )


## Getting Data

In [ ]:
showUsers()

In [ ]:
dummySession= {'userid': 64801 }

In [ ]:
def addMissing():
   

    # 1. Add the dummy user
    now = int(datetime.now().timestamp())
    globalStore.users.insert(dict(
        username="dummy", email="dummy@test.com", password="test",
        created=now, sessionID="", activeWorld=0, id=64801
    ))

    # 2. Create a world for them
    active = globalStore.create_game(64801, template_name="bayArea")

    # 3. Recruit a Knight into the first settlement of the first kingdom
    board = active.board
    kingdom = board.kingdoms[0]
    settlement = kingdom.settlements[0]
    settlement.db = globalStore  # wire up DB so save works

    knight = settlement.recruit(PieceType.KNIGHT, flag=kingdom.flag)
    knight.save(db=globalStore, world_id=active.world_id, kingdom_id=kingdom.countryId)

    print(f"User: 64801")
    print(f"World: {active.world_id}")
    print(f"Kingdom: {kingdom.countryName} (id={kingdom.countryId})")
    print(f"Settlement: {settlement.name} @ hex {settlement.location}")
    print(f"Knight: {knight.id[:12]}… facing={knight.facing} atk={knight.attack_strength} mov={knight.move_strength}")


In [ ]:
#addMissing()

In [ ]:
def showPieces():
    users_df = pd.DataFrame(globalStore.pieces())
    print("pieces:")
    print(users_df)

In [ ]:
showPieces()

In [ ]:
def showActiveSettlements(session):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}")
    active = globalStore.active_board(uid)
    ret = []
    for country in active.board.kingdoms:
        ret.extend( country.settlements)
    return ret


In [ ]:
showActiveSettlements(dummySession)

In [ ]:
def listSettlements():
    users_df = pd.DataFrame(globalStore.settlements())
    print("pieces:")
    print(users_df)

In [ ]:
listSettlements()

In [ ]:
dummySettlment = "7ba2cc4b-97fd-4682-8d34-e4b9ea06cd72"

In [ ]:
@rt("/piece_map/{id}")
def piece_map(session, id: str, rings: int = None):
    """Map partial — renders zoomed terrain with active overlays."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game", id="map")

    if rings is not None:
        rings = max(3, min(12, rings))
        session['settlement_rings'] = rings
    else:
        rings = session.get('settlement_rings', 5)

    overlays = get_overlays(session)

    try:
        place = globalStore.piece_from_id(id)
        board = active.board
        terrain = board.terrain

        region = place.region(terrain.hexGrid, rings=rings)
        if len(region.hexes) == 0:
            return P("Settlement region is empty", id="map")

        result = active.cover.zoom_region_fast(region, compute_weather=True)
        zoomed = result.terrain
        zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
        zoomed.hexGrid.builder.layers = []
        zoomed.colorMap()
        zoomed.hexGrid.update()
        c2f = result.invert_mapper()

        piece_attrs = {
            'hx-get': f'/piece_detail/{id}',
            'hx-target': '#right-panel',
            'hx-swap': 'innerHTML',
            'style': 'cursor: pointer;',
        }
        zoomed.apply_overlays(result, board, c2f, overlays, piece_attrs)

        return Div(
            HexTouchMap(zoomed.hexGrid, cls="w-full h-full"),
            id="map"
        )
    except Exception as e:
        logging.error(f"settlement_map FAILED: {e}", exc_info=True)
        return P(f"Error: {e}", id="map")


We have done so much since we first wrote this? THoughts on using a piece board. we definitely want to show the moves and ensure that we can see the piece from permissions

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yeah, there's been a lot of evolution here. Let me lay out where I think things stand and where PieceBoard fits.

**PieceBoard's role today:** It's a self-contained test harness — flat/hilly terrain, no DB, no auth. That's valuable and should stay. But looking at the real pipeline, here are the key gaps and opportunities:

**1. `piece_map` route is broken for pieces.** It calls `place.region()` which doesn't exist on `Piece` — that's a `Settlement` method. For a piece you'd want `plan_region()`:

```python
steps, region = piece.plan_region(
    terrain.hexGrid, terrain.elevations,
    num_turns=num_turns, countries=terrain.fields.get("country"),
)
```

That gives you both the simulation steps *and* a padded bounding region to zoom into.

**2. Permission isn't wired into `piece_map`.** You have `_auth_piece` ready to go but `piece_map` still uses raw `ensure_user`. Easy fix — swap in `_auth_piece` and bail with `_DENIED` if it returns None.

**3. The real opportunity: PieceBoard as a "preview sandbox".** Right now you have two rendering paths — PieceBoard for testing, and the real zoom pipeline for production. They both converge on `piece_plan_overlay`, which is great. But you could make PieceBoard the *preview engine* inside the rule editor:

- User opens rule editor → server builds a lightweight PieceBoard from the zoomed region around that piece
- As they drag paths and edit rules, re-simulate on the PieceBoard and return just the `_plan_overlay()` SVG
- On "Apply", persist to DB and re-render the full map

This keeps the expensive zoom+weather computation to the initial load, and makes edit-preview-edit cycles fast.

**4. Multi-piece collision.** `_plan_overlay` runs each piece independently — no shared `occupied` set. For accurate preview you'd want to:
- Gather all pieces in the visible region
- Sort by chess rank (KING → PAWN)
- Feed a shared `occupied` set through in that order

This is straightforward on PieceBoard since it owns the piece list. On the real pipeline, you'd query all pieces in the kingdom/region from DB.

**5. Duplicate pattern defs.** Every `draw_svg`/`draw_ghost` call re-registers the same `<pattern>`. A one-liner guard in `_ensure_pattern` would clean up the SVG output significantly:

```python
pat_name = f"piece_pat_{self.id[:8]}"
if not grid.builder.has_definition(pat_name):
    grid.builder.add_definition(self.flag.flagPattern(pat_name, scale=pat_scale))
```

**My suggestion for next step:** Fix `piece_map` to use `plan_region` + `_auth_piece`, and get a real piece rendering in the game map with permission checks. That closes the loop between the editor and the map view. Then you can layer in the PieceBoard-as-preview-sandbox pattern for the editor.

Which piece of this do you want to tackle first?

ok lets using plan_region and auth_piece - collisions are ok. I have to build out the resolve later. But it might be the case that someone wants to send the same troops to the same hex in a few turns and see who gets their. think about charging up a hill. Can you start to build?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call on charging the hill — multiple pieces converging on the same hex is a legitimate tactical move, and blocking that in preview would be misleading. Here's the build:

```python
def _wire_flag(piece: Piece, board: GameBoard):
    """Attach the correct kingdom flag to a loaded piece."""
    for k in board.kingdoms:
        if k.countryId == piece.owner_id:
            piece.flag = k.flag
            return


def _nearby_pieces(board: GameBoard, target: Piece, region: HexRegion) -> list[Piece]:
    """Other pieces whose current location falls within the visible region."""
    nearby = []
    for kingdom in board.kingdoms:
        for settlement in kingdom.settlements:
            for citizen in settlement.citizens:
                if citizen.id == target.id:
                    continue
                if citizen.location is not None and citizen.location in region.hexes:
                    if not citizen.flag:
                        citizen.flag = kingdom.flag
                    nearby.append(citizen)
    return nearby


def _render_piece_map(session, piece_id: str, num_turns: int = 6):
    """Core renderer: zoomed map centered on a piece with its movement plan.

    No occupied set → overlapping paths are intentionally visible.
    Other pieces in the region shown as static markers at 60% opacity.
    """
    uid = _auth_piece(session, piece_id)
    if uid is None:
        return Div(_DENIED, id="map")

    active = globalStore.active_board(uid)
    if not active:
        return Div(P("No active game"), id="map")

    board     = active.board
    terrain   = board.terrain
    grid      = terrain.hexGrid
    countries = terrain.fields.get("country")

    piece = globalStore.piece_from_id(piece_id)
    _wire_flag(piece, board)

    if piece.location is None or piece.location < 0:
        return Div(P("Piece has no location"), id="map")

    # Simulate on coarse grid — collision-free so convergent paths show
    steps, region = piece.plan_region(
        grid, terrain.elevations,
        num_turns=num_turns,
        countries=countries,
        # occupied deliberately omitted
    )

    if not region.hexes:
        return Div(P("Empty region"), id="map")

    # Zoom into the padded bounding region
    result   = active.cover.zoom_region_fast(region, compute_weather=True)
    zterrain = result.terrain
    zgrid    = zterrain.hexGrid
    zgrid.adjustRadius(grid.radius)
    zgrid.builder.layers = []
    zterrain.colorMap()
    zgrid.update()
    c2f = result.c2f
    N   = len(zgrid.hexes)

    # 1. Country borders
    borders_svg = board.countries_overlay(zterrain, c2f)

    # 2. Other pieces in the area — static position markers
    others_svg = ""
    for other in _nearby_pieces(board, piece, region):
        idx = _map_point(other.location, c2f)
        if 0 <= idx < N:
            others_svg += other.draw_svg(zgrid, hex_idx=idx, opacity=0.6)

    # 3. Target piece's movement plan
    plan_svg = piece_plan_overlay(piece, zgrid, steps=steps, c2f=c2f)

    # Layer order: borders underneath, then other pieces, then plan on top
    zgrid.builder.adjust("borders", borders_svg)
    zgrid.builder.adjust("others",  others_svg)
    zgrid.builder.adjust("plan",    plan_svg)

    return Div(NotStr(zgrid.builder.xml()), id="map")


# ── Routes ────────────────────────────────────────────────────────────────────

@rt("/piece_map/{piece_id}")
def piece_map(session, piece_id: str, num_turns: int = 6):
    """Zoomed map centered on a piece showing its planned movement."""
    return _render_piece_map(session, piece_id, num_turns)


@rt("/piece_plan/{piece_id}")
def piece_plan(session, piece_id: str, num_turns: int = 6):
    """Alias — wired to the piece card's 'Show Plan' button."""
    return _render_piece_map(session, piece_id, num_turns)


@rt("/piece_detail/{piece_id}")
def piece_detail(session, piece_id: str):
    """Full piece stats card for the right panel."""
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED

    active = globalStore.active_board(uid)
    if not active: return P("No active game")

    piece = globalStore.piece_from_id(piece_id)
    _wire_flag(piece, active.board)
    return piece  # Piece.__ft__ renders the card with stats + rules + buttons
```

Key design decisions:

- **No `occupied` set** — pieces can overlap in preview. When you build resolve later, that's where collision matters. Preview is about *intent*.
- **`_nearby_pieces`** scans all kingdoms' settlements, so you see enemy pieces too (at 60% opacity). You'll see them converging on the same hex.
- **Layer order** is borders → other pieces → active plan, so the target piece's path is always on top.
- **`plan_region` with padding=3** gives a comfortable margin around the path. A piece with no rules still gets a centered view.
- **`piece_plan` aliases `piece_map`** so the "Show Plan" button in `Piece.__ft__` just works.

Want to test it with the dummy user's knight, or wire up a quick way to give it some rules first?

In [ ]:
def _wire_flag(piece: Piece, board: GameBoard):
    """Attach the correct kingdom flag to a loaded piece."""
    for k in board.kingdoms:
        if k.countryId == piece.owner_id:
            piece.flag = k.flag
            return


def _nearby_pieces(board: GameBoard, target: Piece, region: HexRegion) -> list[Piece]:
    """Other pieces whose current location falls within the visible region."""
    nearby = []
    for kingdom in board.kingdoms:
        for settlement in kingdom.settlements:
            for citizen in settlement.citizens:
                if citizen.id == target.id:
                    continue
                if citizen.location is not None and citizen.location in region.hexes:
                    if not citizen.flag:
                        citizen.flag = kingdom.flag
                    nearby.append(citizen)
    return nearby


def _render_piece_map(session, piece_id: str, num_turns: int = 6):
    """Core renderer: zoomed map centered on a piece with its movement plan.

    No occupied set → overlapping paths are intentionally visible.
    Other pieces in the region shown as static markers at 60% opacity.
    """
    uid = _auth_piece(session, piece_id)
    if uid is None:
        return Div(_DENIED, id="map")

    active = globalStore.active_board(uid)
    if not active:
        return Div(P("No active game"), id="map")

    board     = active.board
    terrain   = board.terrain
    grid      = terrain.hexGrid
    countries = terrain.fields.get("country")

    piece = globalStore.piece_from_id(piece_id)
    _wire_flag(piece, board)

    if piece.location is None or piece.location < 0:
        return Div(P("Piece has no location"), id="map")

    # Simulate on coarse grid — collision-free so convergent paths show
    steps, region = piece.plan_region(
        grid, terrain.elevations,
        num_turns=num_turns,
        countries=countries,
        # occupied deliberately omitted
    )

    if not region.hexes:
        return Div(P("Empty region"), id="map")

    # Zoom into the padded bounding region
    result   = active.cover.zoom_region_fast(region, compute_weather=True)
    zterrain = result.terrain
    zgrid    = zterrain.hexGrid
    zgrid.adjustRadius(grid.radius)
    zgrid.builder.layers = []
    zterrain.colorMap()
    zgrid.update()
    c2f = result.c2f
    N   = len(zgrid.hexes)

    # 1. Country borders
    borders_svg = board.countries_overlay(zterrain, c2f)

    # 2. Other pieces in the area — static position markers
    others_svg = ""
    for other in _nearby_pieces(board, piece, region):
        idx = _map_point(other.location, c2f)
        if 0 <= idx < N:
            others_svg += other.draw_svg(zgrid, hex_idx=idx, opacity=0.6)

    # 3. Target piece's movement plan
    plan_svg = piece_plan_overlay(piece, zgrid, steps=steps, c2f=c2f)

    # Layer order: borders underneath, then other pieces, then plan on top
    zgrid.builder.adjust("borders", borders_svg)
    zgrid.builder.adjust("others",  others_svg)
    zgrid.builder.adjust("plan",    plan_svg)

    return Div(NotStr(zgrid.builder.xml()), id="map")


# ── Routes ────────────────────────────────────────────────────────────────────

@rt("/piece_map/{piece_id}")
def piece_map(session, piece_id: str, num_turns: int = 6):
    """Zoomed map centered on a piece showing its planned movement."""
    return _render_piece_map(session, piece_id, num_turns)


@rt("/piece_plan/{piece_id}")
def piece_plan(session, piece_id: str, num_turns: int = 6):
    """Alias — wired to the piece card's 'Show Plan' button."""
    return _render_piece_map(session, piece_id, num_turns)


@rt("/piece_detail/{piece_id}")
def piece_detail(session, piece_id: str):
    """Full piece stats card for the right panel."""
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED

    active = globalStore.active_board(uid)
    if not active: return P("No active game")

    piece = globalStore.piece_from_id(piece_id)
    _wire_flag(piece, active.board)
    return piece  # Piece.__ft__ renders the card with stats + rules + buttons


So I am going to start having the flag be what marks settlments so the king returns (good book). So the idea is for a piece map that would show what the piece would do. we should have a number of rounds to show where the piece thinks is going to happen based upon its rules. We are going to want to have that for a group of pieces as well when we have the settlement and kindom maps. We need a region large enough to show all of the moves. so we will have to find the hexes in the fine region and then use the zoom-region to get them which is a smidge complicated

Can we demo piece_detail

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! You've got everything wired up already. Grab the piece ID from your `showPieces()` output and render it:

```python
active = globalStore.active_board(64801)
piece = globalStore.piece_from_id("bbe865d0-1751-4076-a125-1d5e1f804ab5")
_wire_flag(piece, active.board)

# Give it some rules so the card isn't empty
piece.rules = [
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value, Instruction.FORWARD.value,
    Instruction.HARVEST.value,
]
piece.cursor = 0
piece.food = 3.5  # show the food bar partially filled

show(piece)
```

That'll render the full `Piece.__ft__` card with health/food bars, stats grid, instruction table, and the Edit Rules / Show Plan buttons. The buttons won't be live without the server running, but you'll see the layout.

If you want it live with working HTMX buttons:

```python
webMe(piece_detail(dummySession, "bbe865d0-1751-4076-a125-1d5e1f804ab5"))
```

Though that path hits `_auth_piece` which checks the DB, so the first approach is quicker for a visual check.

@patch
def __ft__(self: Piece):
    """Compact vertical card for side-panel display."""
    ptype = self.piece_type.name.title()
    short_id = self.id[:12] + "…"
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # ── Piece icon as inline SVG with flag pattern ──
    icon_size = 60
    icon_grid = HexGrid.centered(rings=0, radius=icon_size * 0.6,
                                  style=StyleCSS("_ic", fill="transparent"))
    icon_svg = self.draw_svg(icon_grid, hex_idx=icon_grid.middle,
                              scale=(icon_size * 0.5) / 22.5) if self.flag else ""
    pad = icon_size * 0.8
    icon_html = (f'<svg width="{icon_size}" height="{icon_size}" '
                 f'viewBox="{-pad} {-pad} {icon_size+pad*2} {icon_size+pad*2}" '
                 f'xmlns="http://www.w3.org/2000/svg">'
                 f'{icon_grid.builder.defs_xml()}'
                 f'{icon_svg}</svg>') if icon_svg else _PIECE_ICONS.get(self.piece_type, "?")

    # ── Header: big icon + name/facing ──
    header = DivCentered(
        NotStr(icon_html) if icon_svg else P(icon_html, cls="text-5xl"),
        H4(ptype, cls="font-bold mt-1"),
        P(f"{short_id}  •  Facing {facing_arrow}", cls=TextPresets.muted_sm),
        cls="text-center",
    )

    # ── Health + Food bars ──
    bars = Div(
        StatBar("Health", self.health, self.max_health),
        StatBar("Food", self.food, self.food_capacity),
        cls="space-y-1",
    )

    # ── Stats as a single tight row ──
    stats = [
        ("⚔", self.attack_strength), ("🏃", self.move_strength),
        ("🌾", self.harvest_strength), ("👁", self.sight),
    ]
    stat_row = DivFullySpaced(
        *[Div(
            P(f"{icon} {val}", cls="text-sm font-bold text-center"),
          ) for icon, val in stats],
        cls="py-1",
    )

    # ── Rules list ──
    rules_section = InstructionList(self.rules, self.cursor, self.patrol)

    # ── Action buttons ──
    actions = DivFullySpaced(
        Button("Edit Rules",
               hx_get=f"/piece_rules/{self.id}",
               hx_target="#right-panel",
               cls=ButtonT.primary + " " + ButtonT.sm),
        Button("Show Plan",
               hx_get=f"/piece_plan/{self.id}",
               hx_target="#map",
               cls=ButtonT.sm),
    )

    return Card(
        header,
        bars,
        stat_row,
        Divider(),
        rules_section,
        Divider(),
        actions,
        id=f"piece-card-{self.id[:8]}",
        cls="space-y-2",
    )


### FastHTML

where is  _PIECE_ICONS defined?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

It's in the **§4 Piece extensions** code cell (`_53c77b7f`), right next to `_PIECE_RENDERERS`:

```python
_PIECE_ICONS = {
    PieceType.PAWN:   "♟", PieceType.KNIGHT: "♞", PieceType.ROOK:   "♜",
    PieceType.BISHOP: "♝", PieceType.QUEEN:  "♛", PieceType.KING:   "♚",
}
```

In [ ]:
!cat ../../HexMagic/game/flag.py

In [ ]:
@patch
def __ft__(self: Piece):
    """Narrow vertical layout for w-64 sidebar panel."""
    ptype = self.piece_type.name.title()
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # ── Flag-patterned piece icon (80px) ──
    icon_size = 80
    icon_el = P(self.piece_type.icon, cls="text-5xl")  # fallback: unicode glyph from PieceType.icon

    if self.flag:
        builder = SVGBuilder()
        builder.width = icon_size
        builder.height = icon_size
        center = MapCord(icon_size / 2, icon_size / 2)
        scale = (icon_size * 0.55) / 22.5
        svg = self.flag.draw_piece(
            self.piece_type, center, builder,
            scale=scale, size='large',
            piece_id=f"piece_icon_{self.id[:8]}",
            layer="piece",
        )
        if svg:
            icon_el = NotStr(builder.xml())

    # ── Name / type / birth ──
    display_name = self.name if self.name else ptype
    subtitle = ptype
    if self.name:
        subtitle = f"{ptype} • b.{self.birth_year}" if self.birth_year else ptype

    # ── Stats row ──
    stats = [("⚔", self.attack_strength, "Atk"),
             ("🏃", self.move_strength,   "Spd"),
             ("🌾", self.harvest_strength, "Harv"),
             ("👁", self.sight,            "Sight")]
    stat_row = DivFullySpaced(
        *[Div(
            P(f"{ic} {v}", cls="text-sm font-bold text-center"),
            P(label, cls="text-[10px] text-center opacity-50"),
          ) for ic, v, label in stats],
    )

    # ── Rules — compact badge strip ──
    rules_el = self.instructions.compact(self.id)

    return Div(
        DivCentered(icon_el, cls="pt-2"),
        DivCentered(
            Strong(display_name),
            P(subtitle, cls=TextPresets.muted_sm),
        ),
        P(f"Facing {facing_arrow}", cls="text-center text-xs opacity-60"),

        StatBar("Health", self.health, self.max_health),
        StatBar("Food",   self.food,   self.food_capacity),

        stat_row,

        Divider(),
        rules_el,

        Divider(),
        DivVStacked(
            Button("Edit Rules",
                   hx_get=f"/piece_rules/{self.id}",
                   hx_target="#right-panel", hx_swap="innerHTML",
                   cls=ButtonT.primary + " " + ButtonT.sm + " w-full"),
            Button("Show Plan",
                   hx_get=f"/piece_plan/{self.id}",
                   hx_target="#map", hx_swap="innerHTML",
                   cls=ButtonT.sm + " w-full"),
            cls="space-y-1",
        ),

        id=f"piece-card-{self.id[:8]}",
        cls="space-y-2 p-3",
    )


In [ ]:
active = globalStore.active_board(64801)
piece = globalStore.piece_from_id("bbe865d0-1751-4076-a125-1d5e1f804ab5")
_wire_flag(piece, active.board)

# Give it some rules so the card isn't empty
piece.rules = [
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value, Instruction.FORWARD.value,
    Instruction.HARVEST.value,
]
piece.cursor = 0
piece.food = 3.5  # show the food bar partially filled

show(piece)


In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

# Three tiers with their typical scales
tiers = [
    ('board',  0.5,  'solid fill'),
    ('list',   1.5,  'simple pattern'),
    ('large',  3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx = 80 + i * spacing
    cy = 90
    pid = f"queen_{size}_{i}"

    # Draw backdrop circle scaled to piece size
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    # Draw the queen using draw_piece
    flag.draw_piece(
        PieceType.ROOK, MapCord(cx, cy), canvas,
        scale=scale, size=size, piece_id=pid, layer=f"queen_{i}",
    )

    # Labels
    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{size} (×{scale})</text>'
        f'<text x="{cx}" y="{cy + r + 48}" text-anchor="middle" '
        f'font-size="11" font-family="sans-serif" fill="#666">{desc}</text>')

# Title
canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="20" text-anchor="middle" '
    f'font-size="16" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name} — Queen at 3 tiers (pattern #{flag.patternIndex})</text>')

canvas.show()

In [ ]:
#| export
@patch
def badge(piece: Piece, size: int = 32) -> FT:
    """Colored circle with unicode chess glyph — readable at any size."""
    icon = piece.piece_type.icon
    if not piece.flag:
        return P(icon, cls="text-lg")
    bg, fg = piece.flag.primary, piece.flag.comp
    return Div(
        P(icon, style=f"color:{fg}; font-size:{size*0.55}px; line-height:1;"),
        style=(f"width:{size}px; height:{size}px; border-radius:50%; "
               f"background:{bg}; display:flex; align-items:center; "
               f"justify-content:center; border:2px solid {fg};"),
    )


In [ ]:
#| export
def PieceList(pieces: list[Piece], 
              hx_get_prefix: str = "/piece_detail",
              hx_target: str = "#right-panel",
              hx_swap: str = "innerHTML",
              icon_size: int = 32,
              selected_id: str = "",
              **kwargs):
    """Compact clickable piece roster for sidebar panels.
    
    Each row: [flag-colored badge] Name  b.Year  ♥Health
    Clicking a row loads that piece's detail card.
    """
    if not pieces:
        return P("No pieces", cls="text-sm opacity-50")

    rows = []
    for p in pieces:
        icon_el = p.badge(icon_size)

        # Health as colored text
        hp_pct = (p.health / p.max_health * 100) if p.max_health else 0
        hp_color = ("text-green-500" if hp_pct > 60
                    else "text-yellow-500" if hp_pct > 30
                    else "text-red-500")

        name = p.name or p.piece_type.name.title()
        year = f"b.{p.birth_year}" if p.birth_year else ""

        is_selected = p.id == selected_id
        sel_cls = ("bg-primary/10 border-primary" if is_selected
                   else "border-transparent hover:bg-base-300")

        row = Div(
            Div(icon_el, cls="flex-shrink-0"),
            Div(
                P(name, cls="text-sm font-bold truncate"),
                P(year, cls="text-[10px] opacity-50") if year else None,
                cls="flex-1 min-w-0",
            ),
            P(f"♥{p.health}", cls=f"text-xs font-mono {hp_color} flex-shrink-0"),

            hx_get=f"{hx_get_prefix}/{p.id}",
            hx_target=hx_target,
            hx_swap=hx_swap,
            cls=f"flex items-center gap-2 px-2 py-1 rounded cursor-pointer border {sel_cls}",
        )
        rows.append(row)

    return Div(*rows, cls="space-y-1", **kwargs)


Can you rewrite PieceList since we no longer have _PIECE_ICONS_ and are using country flag calls.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure — `PieceType` already has the `.icon` property, and you just built `piece.badge()` which does all the flag-colored work. This simplifies a lot:

```python
def PieceList(pieces: list[Piece], 
              hx_get_prefix: str = "/piece_detail",
              hx_target: str = "#right-panel",
              hx_swap: str = "innerHTML",
              icon_size: int = 32,
              selected_id: str = "",
              **kwargs):
    """Compact clickable piece roster for sidebar panels.
    
    Each row: [flag-colored badge] Name  b.Year  ♥Health
    Clicking a row loads that piece's detail card.
    """
    if not pieces:
        return P("No pieces", cls="text-sm opacity-50")

    rows = []
    for p in pieces:
        icon_el = p.badge(icon_size)

        # Health as colored text
        hp_pct = (p.health / p.max_health * 100) if p.max_health else 0
        hp_color = ("text-green-500" if hp_pct > 60
                    else "text-yellow-500" if hp_pct > 30
                    else "text-red-500")

        name = p.name or p.piece_type.name.title()
        year = f"b.{p.birth_year}" if p.birth_year else ""

        is_selected = p.id == selected_id
        sel_cls = ("bg-primary/10 border-primary" if is_selected
                   else "border-transparent hover:bg-base-300")

        row = Div(
            Div(icon_el, cls="flex-shrink-0"),
            Div(
                P(name, cls="text-sm font-bold truncate"),
                P(year, cls="text-[10px] opacity-50") if year else None,
                cls="flex-1 min-w-0",
            ),
            P(f"♥{p.health}", cls=f"text-xs font-mono {hp_color} flex-shrink-0"),

            hx_get=f"{hx_get_prefix}/{p.id}",
            hx_target=hx_target,
            hx_swap=hx_swap,
            cls=f"flex items-center gap-2 px-2 py-1 rounded cursor-pointer border {sel_cls}",
        )
        rows.append(row)

    return Div(*rows, cls="space-y-1", **kwargs)
```

The main changes:
- Replaced the entire manual SVG-icon block with `p.badge(icon_size)` — one line instead of ~15
- `badge()` uses `p.piece_type.icon` internally, so no `_PIECE_ICONS` dependency
- Dropped the `HexGrid.centered` / `draw_svg` / viewBox gymnastics — `badge()` handles all of that as a simple CSS circle with the unicode glyph

You'll also want to update `badge()` itself to use `piece.piece_type.icon` instead of the dict lookup:

```python
@patch
def badge(piece: Piece, size: int = 32) -> FT:
    """Colored circle with unicode chess glyph — readable at any size."""
    icon = piece.piece_type.icon
    if not piece.flag:
        return P(icon, cls="text-lg")
    bg, fg = piece.flag.primary, piece.flag.comp
    return Div(
        P(icon, style=f"color:{fg}; font-size:{size*0.55}px; line-height:1;"),
        style=(f"width:{size}px; height:{size}px; border-radius:50%; "
               f"background:{bg}; display:flex; align-items:center; "
               f"justify-content:center; border:2px solid {fg};"),
    )
```

# Build a board with a few varied pieces
board = PieceBoard.hilly(rings=5, radius=22, seed=7)
grid  = board.grid
mid   = grid.middle

scout = board.add_piece(
    mid, piece_type=PieceType.KNIGHT, facing=3, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
    ]),
)

worker = board.add_piece(
    mid - 3, piece_type=PieceType.PAWN, facing=4, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.HARVEST.value, Instruction.HARVEST.value,
    ]),
)

guard = board.add_piece(
    mid + 7, piece_type=PieceType.ROOK, facing=0, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value,
        Instruction.DEFEND.value, Instruction.DEFEND.value,
    ]),
)

bishop = board.add_piece(
    mid + grid.nCols * 2, piece_type=PieceType.BISHOP, facing=5, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_L.value, Instruction.SETTLE.value,
    ]),
)

# Give them some flavor
scout.food = 7.0
worker.food = 2.0; worker.health = 85
guard.food = 9.0; guard.health = 40
bishop.food = 5.5

# Render side by side
show(Div(
    # Left panel — piece roster
    Div(
        H5("Units", cls="font-bold mb-2"),
        PieceList(board.pieces, 
                  hx_get_prefix="/piece_detail",
                  hx_target="#demo-detail",
                  selected_id=scout.id),
        Divider(),
        Div(id="demo-detail"),
        cls="w-64 bg-base-200 p-3 flex-shrink-0 overflow-y-auto",
    ),

    # Main area — board with plan overlay
    Div(
        NotStr(board.render(show_plan=True, num_turns=3)),
        cls="flex-1 overflow-auto",
    ),

    cls="flex", style="height:500px;",
))


Great demo, can we do it with two different flags so we can see what it looks like? Maybe we should group the piece list by country (and then sort by name)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Two things needed: a grouped variant of `PieceList`, and giving the flags distinct names in the demo.

```python
def GroupedPieceList(pieces: list[Piece],
                     country_names: dict[int, str] = None,
                     country_flags: dict[int, 'CountryFlag'] = None,
                     hx_get_prefix: str = "/piece_detail",
                     hx_target: str = "#right-panel",
                     hx_swap: str = "innerHTML",
                     icon_size: int = 32,
                     selected_id: str = "",
                     **kwargs):
    """Piece roster grouped by country, sorted by name within each group."""
    if not pieces:
        return P("No pieces", cls="text-sm opacity-50")

    country_names = country_names or {}
    country_flags = country_flags or {}

    # Group by owner_id
    groups = {}
    for p in pieces:
        groups.setdefault(p.owner_id, []).append(p)

    # Sort each group by name
    for cid in groups:
        groups[cid].sort(key=lambda p: (p.name or p.piece_type.name).lower())

    sections = []
    for cid in sorted(groups):
        name = country_names.get(cid, f"Country {cid}")
        flag = country_flags.get(cid)

        # Country header — colored dot + name
        dot_color = flag.primary if flag else "#888"
        header = DivLAligned(
            Span(style=f"width:10px; height:10px; border-radius:50%; "
                       f"background:{dot_color}; display:inline-block; flex-shrink:0;"),
            P(name, cls="text-xs font-bold uppercase tracking-wide opacity-70"),
            cls="gap-2 pt-2 pb-1 px-1",
        )

        roster = PieceList(
            groups[cid],
            hx_get_prefix=hx_get_prefix,
            hx_target=hx_target,
            hx_swap=hx_swap,
            icon_size=icon_size,
            selected_id=selected_id,
        )
        sections.append(Div(header, roster))

    return Div(*sections, cls="space-y-2", **kwargs)
```

And the demo with named countries:

```python
board = PieceBoard.hilly(rings=5, radius=22, seed=7)
grid  = board.grid
mid   = grid.middle

# Country 1 — green-ish flag
scout = board.add_piece(
    mid, piece_type=PieceType.KNIGHT, facing=3, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
    ]),
)
worker = board.add_piece(
    mid - 3, piece_type=PieceType.PAWN, facing=4, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.HARVEST.value, Instruction.HARVEST.value,
    ]),
)
queen = board.add_piece(
    mid + 1, piece_type=PieceType.QUEEN, facing=2, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_L.value, Instruction.FORWARD.value,
    ]),
)

# Country 2 — red-ish flag
guard = board.add_piece(
    mid + 7, piece_type=PieceType.ROOK, facing=0, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value,
        Instruction.DEFEND.value, Instruction.DEFEND.value,
    ]),
)
bishop = board.add_piece(
    mid + grid.nCols * 2, piece_type=PieceType.BISHOP, facing=5, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_L.value, Instruction.SETTLE.value,
    ]),
)

# Flavor
scout.food = 7.0
worker.food = 2.0; worker.health = 85
queen.food = 8.0
guard.food = 9.0; guard.health = 40
bishop.food = 5.5

# Country metadata
country_names = {1: "Thornwall Keep", 2: "Red Basin"}

show(Div(
    # Left panel — grouped piece roster
    Div(
        H5("Units", cls="font-bold mb-2"),
        GroupedPieceList(
            board.pieces,
            country_names=country_names,
            country_flags=board.flags,
            hx_get_prefix="/piece_detail",
            hx_target="#demo-detail",
            selected_id=scout.id,
        ),
        Divider(),
        Div(id="demo-detail"),
        cls="w-64 bg-base-200 p-3 flex-shrink-0 overflow-y-auto",
    ),

    # Main area — board with plan overlay
    Div(
        NotStr(board.render(show_plan=True, num_turns=3)),
        cls="flex-1 overflow-auto",
    ),

    cls="flex", style="height:550px;",
))
```

Each country gets a colored dot header matching its flag's primary color, with pieces alphabetically sorted underneath. The two flag palettes from `CountryFlag.seaborn("husl")` should give you visually distinct piece icons — country 1 and country 2 will have different pattern fills on their chess glyphs.

In [ ]:
def GroupedPieceList(pieces: list[Piece],
                     country_names: dict[int, str] = None,
                     country_flags: dict[int, 'CountryFlag'] = None,
                     hx_get_prefix: str = "/piece_detail",
                     hx_target: str = "#right-panel",
                     hx_swap: str = "innerHTML",
                     icon_size: int = 32,
                     selected_id: str = "",
                     **kwargs):
    """Piece roster grouped by country, sorted by name within each group."""
    if not pieces:
        return P("No pieces", cls="text-sm opacity-50")

    country_names = country_names or {}
    country_flags = country_flags or {}

    # Group by owner_id
    groups = {}
    for p in pieces:
        groups.setdefault(p.owner_id, []).append(p)

    # Sort each group by name
    for cid in groups:
        groups[cid].sort(key=lambda p: (p.name or p.piece_type.name).lower())

    sections = []
    for cid in sorted(groups):
        name = country_names.get(cid, f"Country {cid}")
        flag = country_flags.get(cid)

        # Country header — colored dot + name
        dot_color = flag.primary if flag else "#888"
        header = DivLAligned(
            Span(style=f"width:10px; height:10px; border-radius:50%; "
                       f"background:{dot_color}; display:inline-block; flex-shrink:0;"),
            P(name, cls="text-xs font-bold uppercase tracking-wide opacity-70"),
            cls="gap-2 pt-2 pb-1 px-1",
        )

        roster = PieceList(
            groups[cid],
            hx_get_prefix=hx_get_prefix,
            hx_target=hx_target,
            hx_swap=hx_swap,
            icon_size=icon_size,
            selected_id=selected_id,
        )
        sections.append(Div(header, roster))

    return Div(*sections, cls="space-y-2", **kwargs)


In [ ]:
board = PieceBoard.hilly(rings=5, radius=22, seed=7)
grid  = board.grid
mid   = grid.middle

# Country 1 — green-ish flag
scout = board.add_piece(
    mid, piece_type=PieceType.KNIGHT, facing=3, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
    ]),
)
worker = board.add_piece(
    mid - 3, piece_type=PieceType.PAWN, facing=4, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.HARVEST.value, Instruction.HARVEST.value,
    ]),
)
queen = board.add_piece(
    mid + 1, piece_type=PieceType.QUEEN, facing=2, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_L.value, Instruction.FORWARD.value,
    ]),
)

# Country 2 — red-ish flag
guard = board.add_piece(
    mid + 7, piece_type=PieceType.ROOK, facing=0, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value,
        Instruction.DEFEND.value, Instruction.DEFEND.value,
    ]),
)
bishop = board.add_piece(
    mid + grid.nCols * 2, piece_type=PieceType.BISHOP, facing=5, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_L.value, Instruction.SETTLE.value,
    ]),
)

# Flavor
scout.food = 7.0
worker.food = 2.0; worker.health = 85
queen.food = 8.0
guard.food = 9.0; guard.health = 40
bishop.food = 5.5

# Country metadata
country_names = {1: "Thornwall Keep", 2: "Red Basin"}

show(Div(
    # Left panel — grouped piece roster
    Div(
        H5("Units", cls="font-bold mb-2"),
        GroupedPieceList(
            board.pieces,
            country_names=country_names,
            country_flags=board.flags,
            hx_get_prefix="/piece_detail",
            hx_target="#demo-detail",
            selected_id=scout.id,
        ),
        Divider(),
        Div(id="demo-detail"),
        cls="w-64 bg-base-200 p-3 flex-shrink-0 overflow-y-auto",
    ),

    # Main area — board with plan overlay
    Div(
        NotStr(board.render(show_plan=True, num_turns=3)),
        cls="flex-1 overflow-auto",
    ),

    cls="flex", style="height:550px;",
))


I think there need to be better display of style at this size. There is much black for strokes and only a slight color. we probably need to think of a small_rep since these patterns don't work as well at this scale. mabye the primary should be countryFlag instead of black. Thoughts?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're spot on. The problem is twofold:

1. **Pattern fill is unreadable at 32px** — those circles/shapes designed for scale 0.1 become noise at this size
2. **Dark strokes dominate** — the SVG chess paths are mostly thin shapes, so at 32px it's 80% stroke and 20% fill

A few options, roughly from simplest to most polished:

**Option A: Solid primary fill + comp stroke** — cheapest fix. At small scale, skip the pattern entirely and use `flag.primary` as a solid fill with `flag.comp` as the stroke. This makes each country's pieces instantly distinguishable by color:

```python
@patch
def draw_svg_small(self: Piece, grid: HexGrid, hex_idx: int = None,
                   scale: float = None, opacity: float = 1.0) -> str:
    """Simplified piece glyph for roster-sized rendering — solid colors, no pattern."""
    idx = hex_idx if hex_idx is not None else self.location
    if idx is None or idx < 0 or idx >= len(grid.hexes): return ""
    if not self.flag: return ""

    center = grid.hexes[idx].center
    renderer = _PIECE_RENDERERS.get(self.piece_type, 'pawnPiece')
    if scale is None:
        scale = (grid.radius * 0.75) / 22.5

    pid = f"piece_sm_{self.id[:8]}"
    return getattr(self.flag, renderer)(
        MapCord(center.x, center.y),
        scale=scale,
        piece_id=pid,
        fill=self.flag.primary,       # solid fill
        stroke=self.flag.comp,        # complement stroke
    )
```

**Option B: Colored badge with unicode glyph** — skip SVG chess paths entirely at roster size. Much cleaner at 32px:

```python
def _piece_badge(piece: Piece, size: int = 32) -> FT:
    """Colored circle with unicode chess glyph — readable at any size."""
    if not piece.flag:
        return P(_PIECE_ICONS.get(piece.piece_type, "?"), cls="text-lg")
    bg = piece.flag.primary
    fg = piece.flag.comp
    icon = _PIECE_ICONS.get(piece.piece_type, "?")
    return Div(
        P(icon, style=f"color:{fg}; font-size:{size*0.55}px; line-height:1;"),
        style=(f"width:{size}px; height:{size}px; border-radius:50%; "
               f"background:{bg}; display:flex; align-items:center; "
               f"justify-content:center; border:2px solid {fg};"),
    )
```

**My recommendation**: Option B for the roster list (32px), Option A for mid-size contexts (48–64px), and keep the full pattern fill for the map pieces (80px+). Then `PieceList` uses `_piece_badge`:

```python
# In PieceList, replace the icon_el block with:
icon_el = _piece_badge(p, icon_size)
```

This gives you colored circles with ♞♜♝ at roster scale — instantly readable, country-colored, zero SVG complexity. The full fancy flag-patterned pieces stay on the map where they have room to breathe.

Want me to wire up the tiered approach, or start with just Option B in the roster?

Does piece_plan_overlay need to be revised?

#### swap down

Can you give me the full correct versions?

### On Terrain

In [ ]:
showDemo = True

In [ ]:
uid = ensure_user(dummySession)
active = globalStore.active_board(uid)
board, terrain, grid = active.board, active.board.terrain, active.board.terrain.hexGrid

# Find the existing piece (the knight)
kingdom = next(k for k in board.kingdoms
               if any(c for s in k.settlements for c in s.citizens))
piece = next(c for s in kingdom.settlements for c in s.citizens)

# Assign a patrol route: forward-forward-turn, with a harvest stop
piece.instructions = InstructionList([
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.HARVEST.value,
], cursor=0, patrol=True)

piece.patrol = True

# Simulate on the coarse grid
steps, region = piece.plan_region(
    grid, terrain.elevations,
    num_turns=6, padding=4,
    countries=terrain.fields.get("country")
)

print(f"Piece: {piece.piece_type.name} at hex {piece.location}, facing={piece.facing}")
print(f"{len(steps)} steps, {len(region)} hex region\n")
for s in steps:
    tag = " ✘" if s.blocked else ""
    print(f"  {s.step+1:2d}. {s.instruction.name:8s} → hex {s.hex_idx}  f={s.facing}{tag}")

# Zoom into the plan region
result = active.cover.zoom_region_fast(region, compute_weather=True)
zoomed = result.terrain
zoomed.hexGrid.adjustRadius(25)
zoomed.colorMap()
zoomed.hexGrid.update()

# Draw the plan overlay on the zoomed grid
overlay = piece_plan_overlay(piece, zoomed.hexGrid,
                             steps=steps,
                             c2f=result.c2f)

zoomed.hexGrid.builder.adjust("plan", overlay)
if showDemo:
    webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))


### On Text

In [ ]:
uid = ensure_user(dummySession)
active = globalStore.active_board(uid)
board, terrain, grid = active.board, active.board.terrain, active.board.terrain.hexGrid

# Find the knight
kingdom = next(k for k in board.kingdoms
               if any(c for s in k.settlements for c in s.citizens))
piece = next(c for s in kingdom.settlements for c in s.citizens)

# Set up rules: forward-forward-rotate-forward-forward-harvest (patrol + forage)
piece.instructions = InstructionList([
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.HARVEST.value,
], cursor=0, patrol=True)

piece.patrol = True
piece.facing = 0

print(f"Piece: {piece.piece_type.name}, speed={piece.move_strength}, loc={piece.location}")
#print(f"Rules: {[Instruction(r).name for r in piece.rules]}")
print(f"Patrol: {piece.patrol}\n")

# Simulate 5 turns
steps = piece.simulate(grid, terrain.elevations, num_turns=5,
                       countries=terrain.fields.get("country"))

# Print turn-by-turn
by_turn = PieceStep.steps_by_turn(steps)
for t in sorted(by_turn.keys()):
    print(f"── Turn {t+1} ──")
    for s in by_turn[t]:
        moved = f"→ hex {s.hex_idx}" if s.hex_idx != s.prev_idx else f"@ hex {s.hex_idx}"
        rot = f" (auto-rot {s.auto_rotations}×)" if s.auto_rotations else ""
        blk = " BLOCKED" if s.blocked else ""
        print(f"  step {s.step:2d}: {s.instruction.name:8s} {moved}  "
              f"face={s.facing}  cost={s.cost:.0f}  budget={s.budget_remaining:.0f}{rot}{blk}")
    print()

print(f"Total steps: {len(steps)}")
print(f"Final position: hex {steps[-1].hex_idx}, facing {steps[-1].facing}")


In [ ]:
# Re-simulate with the fix applied
steps, region = piece.plan_region(grid, terrain.elevations,
                                  num_turns=5, padding=4,
                                  countries=terrain.fields.get("country"))

print(f"{len(steps)} steps, {len(region)} hex region")

# Zoom
result = active.cover.zoom_region_fast(region, compute_weather=True)
zoomed = result.terrain
zoomed.hexGrid.adjustRadius(25)
zoomed.colorMap()
zoomed.hexGrid.update()

# Draw plan overlay
overlay = piece_plan_overlay(piece, zoomed.hexGrid,
                             steps=steps, c2f=result.c2f)

zoomed.hexGrid.builder.adjust("plan", overlay)
if showDemo:
    webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))
#zoomed.hexGrid.builder.show()


Is there something wrong with the xml?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, there are three clear bugs visible in the output:

**1. Duplicate `<pattern>` defs (5× the same block)**
```xml
<pattern id="piece_pat_bbe865d0" ...>...</pattern>
<pattern id="piece_pat_bbe865d0" ...>...</pattern>
<!-- × 5 -->
```
`add_definition` deduplicates by checking `getattr(item, 'id', None)` — but the pattern's id lives in `item.attributes['id']`, not directly as `item.id`. So the check silently fails every time. Fix:

```python
@patch
def _ensure_pattern(self: Piece, grid: HexGrid, pat_scale: float = 0.1) -> str:
    if not self.flag: return "#888"
    pat_name = f"piece_pat_{self.id[:8]}"
    # Check attributes dict, not item.id
    already = any(getattr(d, 'attributes', {}).get('id') == pat_name
                  for d in grid.builder.definitions)
    if not already:
        grid.builder.add_definition(self.flag.flagPattern(pat_name, scale=pat_scale))
    return f"url(#{pat_name})"
```

**2. Duplicate ghost SVG IDs**
Every ghost is `id="piece_bbe865d0_ghost"` — invalid SVG. `draw_ghost` needs a unique suffix, e.g. pass the step number through:

```python
@patch
def draw_ghost(self: Piece, grid: HexGrid, hex_idx: int,
               scale: float = None, opacity: float = 0.4,
               suffix_extra: str = "") -> str:
    s = scale or (grid.radius * 0.75) / 22.5
    return self.draw_svg(grid, hex_idx=hex_idx,
                         scale=s * 0.7, opacity=opacity,
                         suffix=f"_ghost{suffix_extra}")
```

Then in `piece_plan_overlay`, pass `suffix_extra=str(step.step)`.

**3. Arrow color is invisible**
`flag.primary` for this kingdom is `#fff5e1` — a cream white. Against light terrain it's essentially invisible. The arrow style should use a darker contrasting colour:

```python
color = piece.flag.primary if piece.flag else "#333"
# If too light, fall back to a dark complement
import colorsys
r,g,b = int(color[1:3],16)/255, int(color[3:5],16)/255, int(color[5:7],16)/255
h,l,s = colorsys.rgb_to_hls(r,g,b)
if l > 0.75:  # very light — use comp or darken
    color = piece.flag.comp if piece.flag else "#333"
```

Or simpler — just always use `flag.comp` (which is designed to contrast with `flag.primary`) for the arrow stroke.

The stacked PAUSE rects (8 identical rects) are technically harmless since the range label `1–5` correctly collapses the display, but it's wasteful — you could skip drawing the glyph when the hex already has one for that turn.

In [ ]:
for s in steps:
    elev = terrain.elevations[s.hex_idx]
    ctry = terrain.fields["country"][s.hex_idx]
    tag = " ✘" if s.blocked else ""
    print(f"  {s.step+1:2d}. hex {s.hex_idx}  elev={elev:.1f}  country={ctry:.0f}{tag}")


## Put it together

In [ ]:
#| export
# ── 4. Route that receives the drag ───────────────────────────

@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, persist, return badge display."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # We need the grid to convert indices → HexPositions.
    # Use a fresh centered grid matching piece_drag_editor's rings/radius.
    rings  = 3
    radius = 35
    dummy_style = StyleCSS("x", fill="white")
    grid = HexGrid.centered(rings, radius=radius, style=dummy_style)

    new_rules = path_to_rules(hex_ids, grid, start_facing=facing)

    # Persist to DB if we can find the piece
    try:
        rec = globalStore.pieces[piece_id]
        if rec:
            existing = [int(x) for x in (rec.get("rules") or "").split(",") if x]
            merged = existing + new_rules   # append; swap for replace if preferred
            globalStore.pieces.upsert(
                {"id": piece_id, "rules": ",".join(str(r) for r in merged)},
                pk="id"
            )
            new_rules = merged
    except Exception:
        pass   # demo mode — just show without saving

    dummy = Piece(id=piece_id, rules=new_rules)
    return rules_display(dummy.rules, piece_id)

In [ ]:
#| export
dummy_rook = Piece(
    id   = "demo-rook-0000-0000-000000000000",
    piece_type = PieceType.ROOK,
    facing     = 4,   # starts facing E
    name = "Lucia"
)

webMe(Div(
    H3("Rook Rule Editor"),
    piece_drag_editor(dummy_rook),
    cls="p-4 space-y-4 bg-slate-900 text-white"
))

Should the piece_drag_editor put a small icon in

In [ ]:
def showActiveSettlements(session):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}")
    active = globalStore.active_board(uid)
    ret = []
    for country in active.board.kingdoms:
        ret.extend( country.settlements)
    return ret

In [ ]:
showActiveSettlements(dummySession)

# Simulate a drag path: center → E → NE → NE
grid = HexGrid.centered(3, radius=35, style=StyleCSS("x", fill="white"))
mid = grid.middle  # 24

# Manually pick a path: center, then one E, then one NE
e_idx  = grid.hexposition_to_index(HexPosition.E, mid)
ne_idx = grid.hexposition_to_index(HexPosition.NE, mid)
ne2    = grid.hexposition_to_index(HexPosition.NE + HexPosition.NE, mid)

drag_path = [mid, e_idx, ne_idx, ne2]
print(f"Drag indices: {drag_path}")

rules = path_to_rules(drag_path, grid, start_facing=4)  # facing E
labels = [_INSTR_LABELS[r][0] for r in rules]
print(f"Rules: {rules}")
print(f"Labels: {labels}")


Is it working?

Yes  wire up the /piece_editor_demo

#| export
def rules_display(rules: list[int], piece_id: str):
    """Compact badge list for a rule sequence, always with correct swap id."""
    target_id = f"rules-{piece_id[:8]}"

    if not rules:
        return Div(
            P("No rules yet — drag a path above", cls="text-sm opacity-50"),
            id=target_id,
        )

    badges = []
    for i, r in enumerate(rules):
        label, color = _INSTR_LABELS.get(r, (str(r), "#888"))
        badges.append(
            Span(label,
                 style=f"background:{color}22; border:1px solid {color}; "
                        "border-radius:4px; padding:2px 6px; font-size:11px; "
                       f"color:{color}; margin:2px; display:inline-block;")
        )

    # Count summary
    forwards = sum(1 for r in rules if r == 5)
    rotations = sum(1 for r in rules if r in (6, 7))
    actions = sum(1 for r in rules if r in (0, 1, 2, 3, 4))

    summary = f"{forwards} moves, {rotations} turns"
    if actions:
        summary += f", {actions} actions"

    return Div(
        P(f"{len(rules)} instructions — {summary}", cls="text-xs opacity-50 mb-1"),
        Div(*badges, style="line-height:2"),
        Pre(",".join(str(r) for r in rules),
            cls="text-xs opacity-40 mt-2 font-mono"),
        id=target_id,
    )


### Revisit

In [ ]:
#| export
@rt("/piece_editor_demo")
def piece_editor_demo(session):
    """Live demo page for the drag-to-rules editor."""
    editor = piece_drag_editor(dummy_rook)
    facing = facing_grid(dummy_rook)
    return Titled("Piece Editor Demo",
        Div(
            H3("Rook Rule Editor"),
            P("Drag across hexes to create movement rules.",
              cls="text-sm opacity-70 mb-4"),
            editor,
            Divider(),
            H4("Facing Selector"),
            facing,
            cls="p-6 space-y-6 max-w-xl mx-auto"
        )
    )


@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, return compact badge strip."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # Reconstruct same grid as piece_drag_editor
    grid = HexGrid.centered(rings=3, radius=35,
                            style=StyleCSS("x", fill="white"))

    new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)

    # For real pieces, merge with existing and persist
    if not piece_id.startswith("demo-"):
        try:
            rec = globalStore.pieces[piece_id]
            if rec:
                existing_str = rec.rules if hasattr(rec, 'rules') else ""
                existing = [int(x) for x in (existing_str or "").split(",") if x]
                new_rules = existing + new_rules
                globalStore.pieces.upsert(
                    {"id": piece_id, "rules": ",".join(str(r) for r in new_rules)},
                    pk="id",
                )
        except Exception as e:
            logging.warning(f"piece_path: DB skip: {e}")

    return InstructionList(new_rules).compact(piece_id)


Is there an interesing demo you think of that shows pieces and drag editing as well as boards?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's a scenario that shows everything off naturally — **"Race to the Summit"**:

Two kingdoms, each with 2–3 pieces starting at opposite edges of a hilly board. A high-elevation hex sits in the middle (the "summit"). The demo lets you:

1. **Click any piece** → right panel shows its card + current rules
2. **"Show Plan"** → overlays dashed movement paths on the board for that piece, with ghost positions per turn
3. **"Edit Rules"** → drag editor opens, you draw a new path, hit Apply → board re-renders showing the updated plan
4. **Both sides' plans visible simultaneously** — you can see if they're converging on the summit from opposite directions

```python
@rt("/summit_demo")
def summit_demo(session):
    board = PieceBoard.hilly(rings=5, radius=22, seed=42)
    grid  = board.grid

    # Find a high hex near center for the summit
    mid = grid.middle
    elevs = board.elevations
    summit_idx = max(range(len(grid.hexes)),
                     key=lambda i: elevs[i] if abs(i - mid) < 15 else 0)

    # Kingdom 1 — NW corner, trying to take the hill
    k1 = [
        board.add_piece(mid - grid.nCols*3 - 2, PieceType.KNIGHT, facing=4,
                        country_id=1,
                        instructions=InstructionList([
                            Instruction.FORWARD.value]*4 + [Instruction.ROT_R.value])),
        board.add_piece(mid - grid.nCols*2 - 3, PieceType.PAWN, facing=3,
                        country_id=1,
                        instructions=InstructionList([
                            Instruction.FORWARD.value,
                            Instruction.FORWARD.value,
                            Instruction.HARVEST.value])),
    ]

    # Kingdom 2 — SE corner, defending the hill
    k2 = [
        board.add_piece(mid + grid.nCols*3 + 2, PieceType.ROOK, facing=1,
                        country_id=2,
                        instructions=InstructionList([
                            Instruction.FORWARD.value]*3 + [Instruction.DEFEND.value])),
        board.add_piece(mid + grid.nCols*2 + 3, PieceType.BISHOP, facing=2,
                        country_id=2,
                        instructions=InstructionList([
                            Instruction.FORWARD.value,
                            Instruction.FORWARD.value,
                            Instruction.ROT_L.value,
                            Instruction.SETTLE.value])),
    ]

    # Mark the summit
    grid.hexes[summit_idx].label = "⛰"
    grid.hexes[summit_idx].labelStyle = "summit-lbl"
    grid.builder.add_style(StyleCSS("summit-lbl", fill="#fff", font_size="14px"))

    board_svg = board.render(show_plan=True, num_turns=4)

    country_names = {1: "Iron Keep", 2: "Ash Basin"}

    return Titled("Race to the Summit",
        Div(
            # Left — roster
            Div(
                H4("⚔ Armies", cls="font-bold mb-2"),
                GroupedPieceList(
                    board.pieces,
                    country_names=country_names,
                    country_flags=board.flags,
                    hx_get_prefix="/piece_detail",
                    hx_target="#right-panel",
                ),
                Divider(),
                P("Click a piece to edit its orders",
                  cls="text-xs opacity-50"),
                cls="w-56 flex-shrink-0 p-3 bg-base-200 overflow-y-auto space-y-2",
            ),

            # Center — board
            Div(
                NotStr(board_svg),
                id="map",
                cls="flex-1 overflow-auto",
            ),

            # Right — piece detail / editor
            Div(
                P("← Select a piece", cls="text-sm opacity-40 p-4"),
                id="right-panel",
                cls="w-64 flex-shrink-0 p-3 bg-base-200 overflow-y-auto",
            ),

            cls="flex gap-0", style="height:600px;",
        )
    )
```

What makes this demo compelling:
- **Narrative tension** — two armies converging, you can see the collision coming
- **Drag editing is motivated** — you *want* to reroute your knight to beat the rook to the top
- **The plan overlay earns its complexity** — ghost positions make the race readable at a glance
- **Both `.render()` and `piece_plan_overlay` are exercised** together with the real HTMX edit loop

The summit marker gives the eye a focal point so the paths feel purposeful rather than random wandering.

In [ ]:
@rt("/summit_demo")
def summit_demo(session):
    board = PieceBoard.hilly(rings=5, radius=22, seed=42)
    grid  = board.grid

    # Find a high hex near center for the summit
    mid = grid.middle
    elevs = board.elevations
    summit_idx = max(range(len(grid.hexes)),
                     key=lambda i: elevs[i] if abs(i - mid) < 15 else 0)

    # Kingdom 1 — NW corner, trying to take the hill
    k1 = [
        board.add_piece(mid - grid.nCols*3 - 2, PieceType.KNIGHT, facing=4,
                        country_id=1,
                        instructions=InstructionList([
                            Instruction.FORWARD.value]*4 + [Instruction.ROT_R.value])),
        board.add_piece(mid - grid.nCols*2 - 3, PieceType.PAWN, facing=3,
                        country_id=1,
                        instructions=InstructionList([
                            Instruction.FORWARD.value,
                            Instruction.FORWARD.value,
                            Instruction.HARVEST.value])),
    ]

    # Kingdom 2 — SE corner, defending the hill
    k2 = [
        board.add_piece(mid + grid.nCols*3 + 2, PieceType.ROOK, facing=1,
                        country_id=2,
                        instructions=InstructionList([
                            Instruction.FORWARD.value]*3 + [Instruction.DEFEND.value])),
        board.add_piece(mid + grid.nCols*2 + 3, PieceType.BISHOP, facing=2,
                        country_id=2,
                        instructions=InstructionList([
                            Instruction.FORWARD.value,
                            Instruction.FORWARD.value,
                            Instruction.ROT_L.value,
                            Instruction.SETTLE.value])),
    ]

    # Mark the summit
    grid.hexes[summit_idx].label = "⛰"
    grid.hexes[summit_idx].labelStyle = "summit-lbl"
    grid.builder.add_style(StyleCSS("summit-lbl", fill="#fff", font_size="14px"))

    board_svg = board.render(show_plan=True, num_turns=4)

    country_names = {1: "Iron Keep", 2: "Ash Basin"}

    return Titled("Race to the Summit",
        Div(
            # Left — roster
            Div(
                H4("⚔ Armies", cls="font-bold mb-2"),
                GroupedPieceList(
                    board.pieces,
                    country_names=country_names,
                    country_flags=board.flags,
                    hx_get_prefix="/piece_detail",
                    hx_target="#right-panel",
                ),
                Divider(),
                P("Click a piece to edit its orders",
                  cls="text-xs opacity-50"),
                cls="w-56 flex-shrink-0 p-3 bg-base-200 overflow-y-auto space-y-2",
            ),

            # Center — board
            Div(
                NotStr(board_svg),
                id="map",
                cls="flex-1 overflow-auto",
            ),

            # Right — piece detail / editor
            Div(
                P("← Select a piece", cls="text-sm opacity-40 p-4"),
                id="right-panel",
                cls="w-64 flex-shrink-0 p-3 bg-base-200 overflow-y-auto",
            ),

            cls="flex gap-0", style="height:600px;",
        )
    )


In [ ]:
!tail -10 base.text

This works and I am so excited. I want to do a commit which means I will loose the prompt history. can you write up where we have been and where we are going so I can put it into a note to come back to.

In [ ]:
homeSettle = [x for x in showActiveSettlements(dummySession) if x.id == "5642d99d-fbbe-4746-81ff-7e8f8552e3ca"][0]

In [ ]:
homeSettle

In [ ]:
knight =  homeSettle.citizens[0]

In [ ]:
knight

We need to adjust knight so that it has its parents_id  (from the '5642d99d-fbbe-4746-81ff-7e8f8552e3ca' and we should save that to the database as well

In [ ]:
knight.parent_id = homeSettle.id
knight.save(db=globalStore, world_id=1, kingdom_id=homeSettle.owner_id)
print(f"Knight parent_id set to {knight.parent_id[:12]}…")


In [ ]:
knight

I think some of the conversation about the simulator wasn't clear when I was doing the prompts. can you write up the approach I am aiming for so I can put it at the top. 

I have done a bunch of reworking on Piece to give them names, birth years, and InstructionList. I think in the long run this will make things easier, but I know that we need to refactor things like __FT__

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good move — having `name`, `birth_year`, and `InstructionList` on the Piece makes the data model much richer. Here's the inventory of what references the old flat fields (`self.rules`, `self.cursor`, `self.patrol`) and needs updating:

**1. `Piece.__ft__`** — the current version uses `self.rules`, `self.cursor`, `self.patrol`:
```python
# Old
rules_el = InstructionList(self.rules, self.cursor, self.patrol).compact(self.id)

# New — it's already an InstructionList
rules_el = self.instructions.compact(self.id)
```

Also add the name/birth_year to the header:
```python
# Replace the short_id line with something like:
DivCentered(
    Strong(f"{self.name}" if self.name else ptype),
    P(f"{ptype} • b.{self.birth_year}" if self.birth_year else ptype,
      cls=TextPresets.muted_sm),
),
```

**2. `simulate`** — references `self.rules` and `self.cursor`:
```python
# Old
if not self.rules or self.location is None:
# ...
cur = ... self.cursor
# ...
if cur >= len(self.rules):
    last = Instruction(self.rules[-1])
# ...
instr = Instruction(self.rules[cur])

# New — go through self.instructions
if not self.instructions.rules or self.location is None:
# ...
cur = self.instructions.cursor
# ...
if cur >= len(self.instructions.rules):
    last = Instruction(self.instructions.rules[-1])
# ...
instr = Instruction(self.instructions.rules[cur])
```

**3. `piece_plan_overlay`** — the check `if not piece.rules` becomes:
```python
# Already fine if simulate handles it, but the standalone overlay function checks too
```

**4. `PieceBoard.add_piece`** — passes `rules=` to the Piece constructor:
```python
# Old
piece = Piece(location=hex_idx, ..., rules=rules or [])

# New — wrap in InstructionList
piece = Piece(location=hex_idx, ...,
              instructions=InstructionList(rules or [], 0, True))
```

**5. `_open_session`** — creates a redundant InstructionList:
```python
# Old
instructions=InstructionList(list(piece.rules), piece.cursor, piece.patrol)

# New — just copy the existing one
instructions=InstructionList(list(piece.instructions.rules),
                             piece.instructions.cursor,
                             piece.instructions.patrol)
```

**6. `piece_drag_editor`** — references `piece.rules`, `piece.cursor`, `piece.patrol`:
```python
# Old
InstructionList(piece.rules, piece.cursor, piece.patrol).compact(piece.id)

# New
piece.instructions.compact(piece.id)
```

**7. The demo cell** where you set rules on the knight:
```python
# Old
piece.rules = [Instruction.FORWARD.value, ...]
piece.patrol = True

# New
piece.instructions = InstructionList([Instruction.FORWARD.value, ...], 0, True)
```

**8. `piece_rule_apply`** — writes flat fields back to DB, still correct because `encode()` already reads from `self.instructions`.

The pattern is consistent: every `self.rules` → `self.instructions.rules`, `self.cursor` → `self.instructions.cursor`, `self.patrol` → `self.instructions.patrol`. And anywhere that *constructs* an `InstructionList` from flat piece fields can just reference `piece.instructions` directly.

Want me to write out the full updated `__ft__` and `simulate` as copy-paste blocks?

please write out __ft__ and simulate as copy-paste blocks?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def __ft__(self: Piece):
    """Narrow vertical layout for w-64 sidebar panel."""
    ptype = self.piece_type.name.title()
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # ── Flag-patterned piece icon (80px) ──
    icon_size = 80
    icon_el = P(_PIECE_ICONS.get(self.piece_type, "?"), cls="text-5xl")

    if self.flag:
        r = icon_size * 0.6
        ig = HexGrid.centered(rings=0, radius=r,
                               style=StyleCSS("_ic", fill="transparent"))
        mid = ig.middle
        svg = self.draw_svg(ig, hex_idx=mid,
                             scale=(icon_size * 0.55) / 22.5)
        if svg:
            cx = ig.hexes[mid].center.x
            cy = ig.hexes[mid].center.y
            m = icon_size * 0.65
            vb = f"{cx-m:.0f} {cy-m:.0f} {m*2:.0f} {m*2:.0f}"
            hdr = ig.builder._header()
            icon_el = NotStr(
                f'<svg width="{icon_size}" height="{icon_size}" '
                f'viewBox="{vb}" xmlns="http://www.w3.org/2000/svg">'
                f'{hdr}{svg}</svg>')

    # ── Name / type / birth ──
    display_name = self.name if self.name else ptype
    subtitle = ptype
    if self.name:
        subtitle = f"{ptype} • b.{self.birth_year}" if self.birth_year else ptype

    # ── Stats row — icon+value on top, label underneath ──
    stats = [("⚔", self.attack_strength, "Atk"),
             ("🏃", self.move_strength,   "Spd"),
             ("🌾", self.harvest_strength, "Harv"),
             ("👁", self.sight,            "Sight")]
    stat_row = DivFullySpaced(
        *[Div(
            P(f"{ic} {v}", cls="text-sm font-bold text-center"),
            P(label, cls="text-[10px] text-center opacity-50"),
          ) for ic, v, label in stats],
    )

    # ── Rules — compact badge strip from InstructionList ──
    rules_el = self.instructions.compact(self.id)

    return Div(
        # Icon + name
        DivCentered(icon_el, cls="pt-2"),
        DivCentered(
            Strong(display_name),
            P(subtitle, cls=TextPresets.muted_sm),
        ),
        P(f"Facing {facing_arrow}", cls="text-center text-xs opacity-60"),

        # Bars
        StatBar("Health", self.health, self.max_health),
        StatBar("Food",   self.food,   self.food_capacity),

        # Stats
        stat_row,

        # Rules
        Divider(),
        rules_el,

        # Buttons
        Divider(),
        DivVStacked(
            Button("Edit Rules",
                   hx_get=f"/piece_rules/{self.id}",
                   hx_target="#right-panel", hx_swap="innerHTML",
                   cls=ButtonT.primary + " " + ButtonT.sm + " w-full"),
            Button("Show Plan",
                   hx_get=f"/piece_plan/{self.id}",
                   hx_target="#map", hx_swap="innerHTML",
                   cls=ButtonT.sm + " w-full"),
            cls="space-y-1",
        ),

        id=f"piece-card-{self.id[:8]}",
        cls="space-y-2 p-3",
    )
```

```python
@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_turns: int = 10,
             countries: np.ndarray = None,
             occupied: set[int] = None) -> list['PieceStep']:
    """Simulate piece over multiple turns using move_strength as per-turn budget.

    Budget rules:
      - FORWARD: costs _move_cost (1 flat, 2 uphill). If can't afford → end turn.
      - ROT_L / ROT_R: free
      - PAUSE / DEFEND / HARVEST / GIVE / SETTLE: cost 1
      - FORWARD into water/edge: auto-rotate left (free, up to 5×). Full circle → stuck, cost 1.
      - FORWARD into occupied hex: pause instead, cost 1.

    End-of-rules:
      - Last instruction was non-movement → pause out remaining budget.
      - Last instruction was movement/rotate → wrap cursor to 0.
    """
    il = self.instructions
    if not il.rules or self.location is None:
        return []

    occupied = set(occupied) if occupied else set()
    steps = []
    pos, fac, cur = self.location, self.facing, il.cursor
    step_n = 0

    for turn in range(num_turns):
        budget = float(self.move_strength)

        while budget > 0:
            # --- End of rules ---
            if cur >= len(il.rules):
                last = Instruction(il.rules[-1])
                if last in _COSTS_ONE:
                    # Idle out the turn
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE,
                                           pos, step_n, turn, budget, 0.0))
                    step_n += 1
                    break
                if il.patrol:
                    cur = 0  # wrap around
                else:
                    break    # one-shot: stop entirely

            instr = Instruction(il.rules[cur])
            prev = pos

            # --- FREE: rotations ---
            if instr in _FREE:
                fac = ((fac - 1) if instr == Instruction.ROT_L
                       else (fac + 1)) % 6
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 0.0, budget))
                step_n += 1; cur += 1
                continue

            # --- COST 1: non-movement actions ---
            if instr in _COSTS_ONE:
                budget -= 1.0
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 1.0, budget))
                step_n += 1; cur += 1
                continue

            # --- FORWARD ---
            auto_rots = 0
            moved = False
            saved_fac = fac

            while auto_rots <= 5:
                d = HexPosition.directions()[fac % 6]
                nbr = grid.hexposition_to_index(d, pos)

                passable = (0 <= nbr < len(elevations)
                            and nbr not in grid.invalidRegion
                            and elevations[nbr] >= 1)
                if passable and countries is not None:
                    passable = countries[nbr] >= 0

                if not passable:
                    fac = (fac - 1) % 6   # free auto-rotate left
                    auto_rots += 1
                    continue

                # Occupied → forced pause
                if nbr in occupied:
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                           step_n, turn, 1.0, budget - 1.0,
                                           blocked=True,
                                           auto_rotations=auto_rots))
                    budget -= 1.0
                    step_n += 1; cur += 1
                    moved = True
                    break

                # Can we afford it?
                cost = _move_cost(elevations, pos, nbr)
                if cost > budget:
                    budget = 0       # turn over, can't afford
                    moved = True
                    break

                # Move
                occupied.discard(pos)
                pos = nbr
                occupied.add(pos)
                budget -= cost
                steps.append(PieceStep(pos, fac, Instruction.FORWARD, prev,
                                       step_n, turn, cost, budget,
                                       auto_rotations=auto_rots))
                step_n += 1; cur += 1
                moved = True
                break

            if not moved:
                # Spun full circle — stuck
                fac = saved_fac
                budget = max(0, budget - 1.0)
                steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                       step_n, turn, 1.0, budget,
                                       blocked=True, auto_rotations=6))
                step_n += 1; cur += 1

    return steps
```

Three changes beyond the `self.rules` → `self.instructions.rules` swap:

1. **Name/birth_year in the header** — shows "Margaret" with "Knight • b.1932" underneath instead of just "Knight" with a truncated UUID
2. **`il.patrol` check at end-of-rules** — the old version always wrapped. Now if `patrol=False` (one-shot), it breaks out of the turn loop when rules are exhausted
3. **`budget = max(0, budget - 1.0)` in the stuck branch** — fixes the negative-budget edge case from the earlier notes

In [ ]:
webMe(summit_demo)